for E2 mini bart

In [ ]:
import torch
import transformers
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Datasets: 4.8.5
Accelerate: 1.15.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:

import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Research")
PROJECT_DIR = REPO_DIR / "projects" / "01-mini-bart"

REPO_URL = "https://github.com/HeavenlyDemon27/Research.git"

def run_command(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

# Clone if missing; otherwise update the existing clone.
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Inspect it before removing or replacing anything."
        )

    run_command(["git", "clone", REPO_URL, str(REPO_DIR)])

else:
    run_command(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_command(["git", "checkout", "main"], cwd=REPO_DIR)
    run_command(
        ["git", "pull", "--ff-only", "origin", "main"],
        cwd=REPO_DIR
    )

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Mini-BART project directory not found: {PROJECT_DIR}"
    )

os.chdir(PROJECT_DIR)

print("\nRepository:", REPO_DIR)
print("Project directory:", Path.cwd())

print("\nCurrent Git commit:")
run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)

print("\nCurrent Git branch:")
run_command(["git", "branch", "--show-current"], cwd=REPO_DIR)


+ git fetch origin
+ git checkout main
+ git pull --ff-only origin main

Repository: /content/Research
Project directory: /content/Research/projects/01-mini-bart

Current Git commit:
+ git rev-parse HEAD

Current Git branch:
+ git branch --show-current


In [ ]:

%pip install -q \
    transformers==5.23.0 \
    datasets==5.2.0 \
    accelerate==1.16.0 \
    sentencepiece \
    pytest


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:

import torch
import transformers
import datasets
import accelerate
import pyarrow

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PyArrow:", pyarrow.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
CUDA: True
Transformers: 5.22.1
Datasets: 4.8.5
Accelerate: 1.15.0
PyArrow: 23.0.1
GPU: Tesla T4


In [ ]:

from pathlib import Path

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research"
)

E2_DIR = DRIVE_PROJECT_DIR / "E2"

PILOT_DIR = E2_DIR / "Pilot"

FINAL_DIR = E2_DIR / "Final"

PILOT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Pilot directory:", PILOT_DIR)
print("Final directory:", FINAL_DIR)

print("Drive setup successful.")


Pilot directory: /content/drive/MyDrive/MiniBART_Research/E2/Pilot
Final directory: /content/drive/MyDrive/MiniBART_Research/E2/Final
Drive setup successful.


In [ ]:

import os
import torch

test_path = PILOT_DIR / "drive_write_test.pt"
temporary_path = str(test_path) + ".tmp"

test_tensor = torch.tensor([10, 20, 30])

torch.save(
    {"test": test_tensor},
    temporary_path
)

os.replace(temporary_path, test_path)

loaded = torch.load(
    test_path,
    map_location="cpu",
    weights_only=True
)

assert torch.equal(loaded["test"], test_tensor)
assert not os.path.exists(temporary_path)

os.remove(test_path)

print("Google Drive checkpoint test PASSED.")


Google Drive checkpoint test PASSED.


# VERIFYING E2 IMPLEMENTATION

In [ ]:

import subprocess

os.chdir(PROJECT_DIR)

test_files = [
    "tests/test_distillation.py",
    "tests/test_kd_setup.py",
    "tests/test_train_kd.py",
    "tests/test_kd_checkpoint.py",
    "tests/test_kd_integration.py",
]

result = subprocess.run(
    ["python", "-m", "pytest", "-q"] + test_files,
    text=True
)

assert result.returncode == 0, "E2 tests failed."

print("E2 test suite passed.")


E2 test suite passed.


# LOAD TOKENIZER AND DATA SET

In [ ]:

import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset

from src.data import (
    load_bart_tokenizer,
    prepare_dataset,
    create_dataloaders,
)

MODEL_NAME = "facebook/bart-large-cnn"
DATASET_NAME = "abisee/cnn_dailymail"
DATASET_VERSION = "3.0.0"

PILOT_TRAIN_SAMPLES = 128
PILOT_VAL_SAMPLES = 32
BATCH_SIZE = 2

print("Loading tokenizer...")

tokenizer = load_bart_tokenizer(MODEL_NAME)

print("Loading training subset...")

train_raw = load_dataset(
    DATASET_NAME,
    DATASET_VERSION,
    split=f"train[:{PILOT_TRAIN_SAMPLES}]"
)

print("Loading validation subset...")

val_raw = load_dataset(
    DATASET_NAME,
    DATASET_VERSION,
    split=f"validation[:{PILOT_VAL_SAMPLES}]"
)

raw_dataset = {
    "train": train_raw,
    "validation": val_raw,
}

print("Training examples:", len(train_raw))
print("Validation examples:", len(val_raw))

assert len(train_raw) == PILOT_TRAIN_SAMPLES
assert len(val_raw) == PILOT_VAL_SAMPLES


Loading tokenizer...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loading training subset...


... (11 outputs left out)

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Loading validation subset...
Training examples: 128
Validation examples: 32


# TOKENIZE AND CREATE DATALOAD

In [ ]:

print("Tokenizing pilot dataset...")

tokenized_dataset = prepare_dataset(
    dataset=raw_dataset,
    tokenizer=tokenizer,
)

# Empty placeholder for the unused test loader.
# No actual CNN/DailyMail test examples are accessed.
tokenized_dataset["test"] = (
    tokenized_dataset["validation"].select([])
)

train_loader, val_loader, unused_test_loader = create_dataloaders(
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("\nDataLoader verification")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(unused_test_loader))

assert len(train_loader) == 64
assert len(val_loader) == 16
assert len(unused_test_loader) == 0

# Verify batch structure
sample_batch = next(iter(val_loader))

for key, value in sample_batch.items():
    print(key, tuple(value.shape), value.dtype)

assert set(sample_batch.keys()) == {
    "input_ids",
    "attention_mask",
    "labels",
}


Tokenizing pilot dataset...


Tokenizing train split:   0%|          | 0/128 [00:00<?, ? examples/s]

Tokenizing validation split:   0%|          | 0/32 [00:00<?, ? examples/s]


DataLoader verification
Training batches: 64
Validation batches: 16
Test batches: 0
input_ids (2, 1024) torch.int64
attention_mask (2, 1024) torch.int64
labels (2, 36) torch.int64


# **INITILIZE E2 MODELS**

In [ ]:

%pip uninstall -y transformers
%pip install --no-cache-dir --force-reinstall transformers==5.23.0


In [ ]:

import sys
import torch
import numpy
import transformers
import huggingface_hub
import tokenizers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("Tokenizers:", tokenizers.__version__)

print("\nCUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from transformers import BartForConditionalGeneration

print("\nSUCCESS: BART import works.")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
NumPy: 2.5.5
Transformers: 5.23.0
HuggingFace Hub: 2.4.0
Tokenizers: 0.23.3

CUDA: True
GPU: Tesla T4

SUCCESS: BART import works.


In [ ]:
!pip check

ipython 7.34.0 requires jedi, which is not installed.
bigframes 2.51.0 has requirement rich<14,>=12.4.4, but you have rich 15.0.0.
numba 0.64.0 has requirement numpy<2.5,>=1.22, but you have numpy 2.5.5.
gcsfs 2025.12.0 has requirement fsspec==2025.12.0, but you have fsspec 2026.9.0.
cudf-cu13 26.6.0 has requirement pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1.


In [ ]:

import torch
from transformers import BartForConditionalGeneration
from transformers import AutoTokenizer

MODEL_NAME = "facebook/bart-large-cnn"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading BART model...")
teacher = BartForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

print("Moving model to GPU...")
teacher = teacher.to("cuda")
teacher.eval()

print("\nModel loaded successfully!")

print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", sum(p.numel() for p in teacher.parameters()))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

assert all(not p.requires_grad for p in teacher.parameters()) is False

# Freeze teacher, as required by E2
for parameter in teacher.parameters():
    parameter.requires_grad = False

assert all(not p.requires_grad for p in teacher.parameters())

print("Teacher frozen successfully.")


Loading tokenizer...


Loading BART model...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Moving model to GPU...

Model loaded successfully!
GPU: Tesla T4
Parameters: 406290432
GPU memory allocated: 0.76 GB
Teacher frozen successfully.


In [ ]:

import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

from transformers import BartForConditionalGeneration

print("BART import successful!")

assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
Transformers: 5.23.0
BART import successful!
GPU: Tesla T4


In [ ]:
import sys
import os
from pathlib import Path

# Find the actual Mini-BART project directory
search_root = Path("/content")

matches = list(
    search_root.rglob("projects/01-mini-bart/src/model.py")
)

if not matches:
    raise FileNotFoundError(
        "Could not locate projects/01-mini-bart/src/model.py. "
        "Check whether the GitHub repository was cloned correctly."
    )

# The project directory is the parent of src/
PROJECT_DIR = matches[0].parent.parent

print("Project directory:", PROJECT_DIR)
print("Model file:", PROJECT_DIR / "src" / "model.py")

# Add project directory to Python's import path
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Set working directory for relative paths
os.chdir(PROJECT_DIR)

# Verify imports
import src.model

print("\nSUCCESS: src.model imported.")

# Now import the required functions
from src.model import (
    load_teacher_model,
    prepare_kd_models,
)

print("SUCCESS: Required model functions imported.")

Project directory: /content/Research/projects/01-mini-bart
Model file: /content/Research/projects/01-mini-bart/src/model.py

SUCCESS: src.model imported.
SUCCESS: Required model functions imported.


In [ ]:
import inspect
from src.model import load_teacher_model, prepare_kd_models

print("load_teacher_model:")
print(inspect.signature(load_teacher_model))
print(inspect.getdoc(load_teacher_model))

print("\nprepare_kd_models:")
print(inspect.signature(prepare_kd_models))
print(inspect.getdoc(prepare_kd_models))

load_teacher_model:
(model_name='facebook/bart-large-cnn')
Load the pretrained BART teacher.

The teacher is used only to generate soft targets during
knowledge distillation.

prepare_kd_models:
(teacher_model)
Prepare the frozen teacher and freshly initialized student for E2.

Returns:
    teacher_model: Frozen pretrained BART teacher.
    student_model: Trainable Mini-BART student.


In [ ]:
import torch

# Reuse the teacher if it was already loaded in an earlier cell.
if "teacher_model" in globals():
    print("Reusing existing teacher_model.")
elif "teacher" in globals():
    teacher_model = teacher
    print("Reusing existing teacher.")
else:
    print("Loading teacher...")
    teacher_model = load_teacher_model()

# Prepare the frozen teacher and fresh student.
teacher_model, student_model = prepare_kd_models(teacher_model)

# Verify model setup.
teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_model.parameters())

teacher_trainable = sum(
    p.numel() for p in teacher_model.parameters() if p.requires_grad
)
student_trainable = sum(
    p.numel() for p in student_model.parameters() if p.requires_grad
)

print("\n========== E2 MODEL VERIFICATION ==========")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Teacher parameters: {teacher_params:,}")
print(f"Student parameters: {student_params:,}")
print(f"Teacher trainable parameters: {teacher_trainable:,}")
print(f"Student trainable parameters: {student_trainable:,}")

print("\nTeacher architecture:")
print("Encoder layers:", teacher_model.config.encoder_layers)
print("Decoder layers:", teacher_model.config.decoder_layers)

print("\nStudent architecture:")
print("Encoder layers:", student_model.config.encoder_layers)
print("Decoder layers:", student_model.config.decoder_layers)

print("\nTeacher device:", next(teacher_model.parameters()).device)
print("Student device:", next(student_model.parameters()).device)

assert teacher_trainable == 0, "Teacher is not fully frozen!"
assert student_trainable > 0, "Student has no trainable parameters!"
assert student_model.config.encoder_layers == 6
assert student_model.config.decoder_layers == 6

print("\nSUCCESS: E2 teacher–student setup verified.")

Reusing existing teacher.

========== E2 MODEL VERIFICATION ==========
GPU available: True
Teacher parameters: 406,290,432
Student parameters: 229,933,056
Teacher trainable parameters: 0
Student trainable parameters: 229,933,056

Teacher architecture:
Encoder layers: 12
Decoder layers: 12

Student architecture:
Encoder layers: 6
Decoder layers: 6

Teacher device: cuda:0
Student device: cpu

SUCCESS: E2 teacher–student setup verified.


In [ ]:

from src.distillation import combined_distillation_loss

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Ensure both models are on the GPU
teacher_model.eval()
student_model.train()

teacher_model.to(device)
student_model.to(device)

smoke_batch = next(iter(val_loader))

smoke_batch = {
    key: value.to(device)
    for key, value in smoke_batch.items()
}

optimizer_test = torch.optim.SGD(
    student_model.parameters(),
    lr=0.0
)

optimizer_test.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()

# Teacher forward pass — no gradients
with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        teacher_outputs = teacher_model(**smoke_batch)

# Student forward pass — gradients enabled
with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    student_outputs = student_model(**smoke_batch)

losses = combined_distillation_loss(
    teacher_logits=teacher_outputs.logits,
    student_logits=student_outputs.logits,
    labels=smoke_batch["labels"],
    temperature=2.0,
    lambda_kd=0.5,
)

assert torch.isfinite(losses["loss"]).item()
assert torch.isfinite(losses["ce_loss"]).item()
assert torch.isfinite(losses["kd_loss"]).item()

losses["loss"].backward()

assert any(
    parameter.grad is not None
    for parameter in student_model.parameters()
)

assert all(
    parameter.grad is None
    for parameter in teacher_model.parameters()
)

print("Total loss:", losses["loss"].item())
print("CE loss:", losses["ce_loss"].item())
print("KD loss:", losses["kd_loss"].item())

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

optimizer_test.zero_grad(set_to_none=True)

del smoke_batch, teacher_outputs, student_outputs, losses
del optimizer_test

torch.cuda.empty_cache()

print("\nGPU smoke test passed.")


Using device: cuda
Total loss: 8.312710762023926
CE loss: 7.540509223937988
KD loss: 1.5444022417068481
Peak GPU memory: 2.8 GB

GPU smoke test passed.


# **LOAD AND PROCESS the pilot DATASET**

In [ ]:
from src.data import (
    load_bart_tokenizer,
    load_cnn_dailymail,
    prepare_dataset,
)

print("Loading BART tokenizer...")
tokenizer = load_bart_tokenizer()

print("Loading CNN/DailyMail 3.0.0...")
raw_dataset = load_cnn_dailymail()

print("Preparing pilot dataset...")
tokenized_dataset = prepare_dataset(
    raw_dataset,
    tokenizer,
    max_train_samples=128,
    max_val_samples=32,
    max_test_samples=0,
)

print("\nDataset ready:")
for split, data in tokenized_dataset.items():
    print(f"{split}: {len(data)} examples")

Loading BART tokenizer...
Loading CNN/DailyMail 3.0.0...
Preparing pilot dataset...


Tokenizing train split:   0%|          | 0/128 [00:00<?, ? examples/s]

Tokenizing validation split:   0%|          | 0/32 [00:00<?, ? examples/s]


Dataset ready:
train: 128 examples
validation: 32 examples
test: 0 examples


In [ ]:
from src.data import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=1,
    num_workers=0,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Device:", device)

Train batches: 128
Validation batches: 32
Device: cuda


# **Create loaders and test teacher–student forward passes**

In [ ]:
import torch

from src.data import create_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Colab GPU is not available!"

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=1,
    num_workers=0,
)

teacher_model.to(device)
student_model.to(device)

teacher_model.eval()
student_model.eval()

batch = next(iter(train_loader))
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

print("Testing teacher and student forward passes...")

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.float16):
        teacher_outputs = teacher_model(**batch)
        student_outputs = student_model(**batch)

print("\nTeacher logits:", teacher_outputs.logits.shape)
print("Student logits:", student_outputs.logits.shape)

assert teacher_outputs.logits.shape == student_outputs.logits.shape
assert teacher_model.training is False

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

del teacher_outputs, student_outputs, batch
torch.cuda.empty_cache()

print("\nSUCCESS: Both models completed forward passes.")

Testing teacher and student forward passes...

Teacher logits: torch.Size([1, 43, 50264])
Student logits: torch.Size([1, 43, 50264])

GPU memory allocated: 1.65 GB

SUCCESS: Both models completed forward passes.


# **LAUNCH E2 PILOT**

In [ ]:
import os
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

from src.train_kd import train_kd_model

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "MiniBART_Research/E2/Pilot"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting E2 pilot training...")
print("Device:", device)
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))

e2_results = train_kd_model(
    teacher=teacher_model,
    student=student_model,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    device=device,
    output_dir=OUTPUT_DIR,
    num_epochs=1,
    temperature=2.0,
    lambda_kd=0.5,
    checkpoint_every_steps=20,
)

print("\n========== E2 PILOT COMPLETE ==========")
print("Best validation loss:", e2_results["best_val_loss"])
print("Global steps:", e2_results["global_step"])
print("Best checkpoint:", e2_results["best_checkpoint"])
print("Latest checkpoint:", e2_results["latest_checkpoint"])

Starting E2 pilot training...
Device: cuda
Training examples: 128
Validation examples: 32

========== E2 Epoch 1/1 ==========


E2 Training:   0%|          | 0/127 [00:00<?, ?it/s]

Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=20
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=40
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=60
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=80
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=100
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=120


E2 Validation:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 1 Results
Train Total: 5.2632 | CE: 4.6240 | KD: 1.2783
Val Total: 4.1452 | CE: 3.6049 | KD: 1.0805
Checkpoint saved | best_kd_checkpoint.pt | next_epoch=1 | step=127
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=1 | step=127

========== E2 PILOT COMPLETE ==========
Best validation loss: 4.145193189382553
Global steps: 127
Best checkpoint: /content/drive/MyDrive/MiniBART_Research/E2/Pilot/best_kd_checkpoint.pt
Latest checkpoint: /content/drive/MyDrive/MiniBART_Research/E2/Pilot/latest_kd_checkpoint.pt


# **Verify and preserve the completed pilot**

In [ ]:

import os
import torch
from pathlib import Path

PILOT_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/Pilot"
)

BEST_PATH = PILOT_DIR / "best_kd_checkpoint.pt"
LATEST_PATH = PILOT_DIR / "latest_kd_checkpoint.pt"

# Verify files exist
assert BEST_PATH.exists(), "Best checkpoint missing!"
assert LATEST_PATH.exists(), "Latest checkpoint missing!"

print("Both checkpoint files exist.")

print("\nCheckpoint sizes:")
print("Best:", round(BEST_PATH.stat().st_size / 1024**2, 2), "MB")
print("Latest:", round(LATEST_PATH.stat().st_size / 1024**2, 2), "MB")

# Load latest checkpoint
latest_ckpt = torch.load(
    LATEST_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nLatest checkpoint information")

print("Epoch:", latest_ckpt["epoch"])
print("Global step:", latest_ckpt["global_step"])
print("Best validation loss:", latest_ckpt["best_val_loss"])

print("\nSaved configuration:")
for key, value in latest_ckpt["config"].items():
    print(f"{key}: {value}")

print("\nCheckpoint components:")
for key in latest_ckpt.keys():
    print("-", key)

# Verify recovery state
required_keys = [
    "student_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
    "sampler_state_dict",
    "rng_state",
]

for key in required_keys:
    assert key in latest_ckpt, f"Missing checkpoint component: {key}"

assert latest_ckpt["epoch"] == 1
assert latest_ckpt["global_step"] == 127

print("\nCheckpoint integrity verification passed.")


Both checkpoint files exist.

Checkpoint sizes:
Best: 877.41 MB
Latest: 2631.9 MB

Latest checkpoint information
Epoch: 1
Global step: 127
Best validation loss: 4.145193189382553

Saved configuration:
temperature: 2.0
lambda_kd: 0.5
num_epochs: 1
objective: cross_entropy_plus_knowledge_distillation
teacher: facebook/bart-large-cnn
student_encoder_layers: 6
student_decoder_layers: 6
learning_rate: 5e-05
weight_decay: 0.01
warmup_ratio: 0.06
max_grad_norm: 1.0

Checkpoint components:
- student_state_dict
- epoch
- global_step
- best_val_loss
- config
- optimizer_state_dict
- scheduler_state_dict
- scaler_state_dict
- sampler_state_dict
- rng_state

Checkpoint integrity verification passed.


# **Save the pilot's research metadata**

In [ ]:

import json
from datetime import datetime, timezone

pilot_history = e2_results["history"]

pilot_record = {
    "experiment": "E2 Mini-BART Knowledge Distillation",
    "run_type": "Pilot",
    "status": "Completed",
    "teacher": "facebook/bart-large-cnn",
    "student_architecture": "6 encoder + 6 decoder layers",
    "dataset": "abisee/cnn_dailymail",
    "dataset_version": "3.0.0",

    "training_examples_reported": 128,
    "validation_examples": 32,

    "epochs": 1,
    "temperature": 2.0,
    "lambda_kd": 0.5,

    "global_steps": e2_results["global_step"],
    "best_validation_loss": e2_results["best_val_loss"],

    "epoch_metrics": pilot_history,

    "latest_checkpoint": str(LATEST_PATH),
    "best_checkpoint": str(BEST_PATH),

    "github_repository": "HeavenlyDemon27/Research",
    "github_branch": "main",
    "github_commit": "de6924c42989a8ba9094b287d14e89feb2ff7674",

    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

metadata_path = PILOT_DIR / "pilot_results.json"

with open(metadata_path, "w") as f:
    json.dump(pilot_record, f, indent=4)

print("Pilot metadata saved:", metadata_path)

print("\nSaved epoch metrics:")
print(json.dumps(pilot_history, indent=4))


Pilot metadata saved: /content/drive/MyDrive/MiniBART_Research/E2/Pilot/pilot_results.json

Saved epoch metrics:
[
    {
        "epoch": 1,
        "train": {
            "loss": 5.263186173176202,
            "ce_loss": 4.62404805468762,
            "kd_loss": 1.2782762186733756,
            "global_step": 127
        },
        "validation": {
            "loss": 4.145193189382553,
            "ce_loss": 3.6049250215291977,
            "kd_loss": 1.0805363859981298
        }
    }
]


# **Investigate the 127 batches**

In [ ]:

print("========== DATALOADER DIAGNOSTICS ==========")

print("Training dataset size:", len(tokenized_dataset["train"]))
print("Training DataLoader batch size:", train_loader.batch_size)
print("Training DataLoader batches:", len(train_loader))

print("Validation dataset size:", len(tokenized_dataset["validation"]))
print("Validation DataLoader batch size:", val_loader.batch_size)
print("Validation DataLoader batches:", len(val_loader))

sampler = train_loader.sampler

print("\nSampler information:")
print("Sampler type:", type(sampler).__name__)
print("Sampler epoch:", sampler.epoch)
print("Sampler position:", sampler.position)
print("Sampler permutation:", sampler.permutation)

import math

expected_batches = math.ceil(
    len(tokenized_dataset["train"]) / train_loader.batch_size
)

print("\nExpected batches:", expected_batches)
print("Actual training steps:", e2_results["global_step"])

print(
    "\nDifference:",
    expected_batches - e2_results["global_step"]
)


========== DATALOADER DIAGNOSTICS ==========
Training dataset size: 128
Training DataLoader batch size: 1
Training DataLoader batches: 128
Validation dataset size: 32
Validation DataLoader batch size: 1
Validation DataLoader batches: 32

Sampler information:
Sampler type: StatefulRandomSampler
Sampler epoch: 1
Sampler position: 0
Sampler permutation: None

Expected batches: 128
Actual training steps: 127

Difference: 1


# **Prepare a small recovery-test dataset**

In [ ]:

from src.data import create_dataloaders

RESUME_TRAIN_SAMPLES = 16
RESUME_VAL_SAMPLES = 4

resume_test_dataset = {
    "train": tokenized_dataset["train"].select(
        range(RESUME_TRAIN_SAMPLES)
    ),

    "validation": tokenized_dataset["validation"].select(
        range(RESUME_VAL_SAMPLES)
    ),

    "test": tokenized_dataset["validation"].select([])
}

# Preserve the batch size used in your pilot.
RESUME_BATCH_SIZE = train_loader.batch_size

resume_train_loader, resume_val_loader, _ = create_dataloaders(
    tokenized_dataset=resume_test_dataset,
    tokenizer=tokenizer,
    batch_size=RESUME_BATCH_SIZE,
    num_workers=0,
)

print("Recovery-test dataset prepared.")

print("Training examples:", len(resume_test_dataset["train"]))
print("Validation examples:", len(resume_test_dataset["validation"]))
print("Batch size:", RESUME_BATCH_SIZE)
print("Training batches:", len(resume_train_loader))
print("Validation batches:", len(resume_val_loader))

assert len(resume_train_loader) > 5


Recovery-test dataset prepared.
Training examples: 16
Validation examples: 4
Batch size: 1
Training batches: 16
Validation batches: 4


# **Start recovery test and interrupt after step 5**

In [ ]:

import os
import gc
import torch
from pathlib import Path

import src.train_kd as train_kd_module

from src.model import prepare_kd_models
from src.train_kd import train_kd_model

RESUME_TEST_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/ResumeTest"
)

RESUME_TEST_DIR.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = (
    RESUME_TEST_DIR / "latest_kd_checkpoint.pt"
)

# Never overwrite a previous recovery test.
assert not RESUME_CHECKPOINT.exists(), (
    "A recovery-test checkpoint already exists. "
    "Inspect it before proceeding."
)

# Initialize a fresh student from the original teacher.
teacher_model, recovery_student = prepare_kd_models(
    teacher_model
)

teacher_model = teacher_model.to(device)
recovery_student = recovery_student.to(device)

class IntentionalPilotPause(Exception):
    pass


original_train_one_epoch = (
    train_kd_module.train_kd_one_epoch
)


def interrupt_after_checkpoint(*args, **kwargs):

    original_callback = kwargs["checkpoint_callback"]

    def save_and_interrupt(step):

        original_callback(step)

        if step == 5:
            raise IntentionalPilotPause(
                "Intentional interruption after step 5."
            )

    kwargs["checkpoint_callback"] = save_and_interrupt

    return original_train_one_epoch(*args, **kwargs)


train_kd_module.train_kd_one_epoch = interrupt_after_checkpoint

try:

    print("Starting recovery-test training...")

    train_kd_model(
        teacher=teacher_model,
        student=recovery_student,
        train_dataloader=resume_train_loader,
        val_dataloader=resume_val_loader,
        device=device,
        output_dir=str(RESUME_TEST_DIR),
        num_epochs=1,
        temperature=2.0,
        lambda_kd=0.5,
        checkpoint_every_steps=5,
    )

except IntentionalPilotPause as error:

    print("\n", error)

finally:

    # Restore original training function.
    train_kd_module.train_kd_one_epoch = (
        original_train_one_epoch
    )


assert RESUME_CHECKPOINT.exists()

recovery_ckpt = torch.load(
    RESUME_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\nRecovery checkpoint verification")

print("Epoch:", recovery_ckpt["epoch"])
print("Global step:", recovery_ckpt["global_step"])

sampler_state = recovery_ckpt["sampler_state_dict"]

print("Sampler position:", sampler_state["position"])
print("Sampler permutation length:", len(sampler_state["permutation"]))

assert recovery_ckpt["epoch"] == 0
assert recovery_ckpt["global_step"] == 5

assert (
    sampler_state["position"]
    < len(sampler_state["permutation"])
)

print("\nMid-epoch checkpoint successfully created.")


Starting recovery-test training...

========== E2 Epoch 1/1 ==========


E2 Training:   0%|          | 0/16 [00:00<?, ?it/s]

Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=5

 Intentional interruption after step 5.

Recovery checkpoint verification
Epoch: 0
Global step: 5
Sampler position: 5
Sampler permutation length: 16

Mid-epoch checkpoint successfully created.


# **Resume from the saved checkpoint**

In [ ]:

import gc
import torch

# Discard the student used before interruption.
del recovery_student

gc.collect()
torch.cuda.empty_cache()

# Initialize another fresh student.
teacher_model, resumed_student = prepare_kd_models(
    teacher_model
)

resumed_student = resumed_student.to(device)

# Recreate the loaders with fresh sampler state.
resumed_train_loader, resumed_val_loader, _ = create_dataloaders(
    tokenized_dataset=resume_test_dataset,
    tokenizer=tokenizer,
    batch_size=RESUME_BATCH_SIZE,
    num_workers=0,
)

print("Resuming from:", RESUME_CHECKPOINT)

resume_test_results = train_kd_model(
    teacher=teacher_model,
    student=resumed_student,
    train_dataloader=resumed_train_loader,
    val_dataloader=resumed_val_loader,
    device=device,
    output_dir=str(RESUME_TEST_DIR),
    num_epochs=1,
    temperature=2.0,
    lambda_kd=0.5,
    resume_from=str(RESUME_CHECKPOINT),
    checkpoint_every_steps=5,
)

print("\n========== RECOVERY TEST COMPLETE ==========")

print("Global steps:", resume_test_results["global_step"])
print("Best validation loss:", resume_test_results["best_val_loss"])

print("Expected total steps:", len(resumed_train_loader))

assert (
    resume_test_results["global_step"]
    == len(resumed_train_loader)
)

assert len(resume_test_results["history"]) == 1

print("\nMid-epoch recovery test passed.")


Resuming from: /content/drive/MyDrive/MiniBART_Research/E2/ResumeTest/latest_kd_checkpoint.pt
Resumed E2 | next_epoch=0 | global_step=5

========== E2 Epoch 1/1 ==========


E2 Training:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:

import torch
import subprocess
import os

print("========== GPU STATUS ==========")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "Reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )

    print(
        "Peak allocated:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

print("\n========== NVIDIA SMI ==========")

subprocess.run(["nvidia-smi"])

print("\n========== SYSTEM MEMORY ==========")

with open("/proc/meminfo") as f:
    for line in f:
        if any(x in line for x in [
            "MemTotal",
            "MemAvailable",
            "SwapTotal",
            "SwapFree"
        ]):
            print(line.strip())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/MiniBART_Research/E2"

for folder in ["Pilot", "ResumeTest"]:
    path = os.path.join(BASE_DIR, folder)

    print(f"\n========== {folder} ==========")

    if not os.path.exists(path):
        print("Folder not found")
        continue

    for filename in os.listdir(path):
        filepath = os.path.join(path, filename)

        if os.path.isfile(filepath):
            size_mb = os.path.getsize(filepath) / (1024 ** 2)
            print(f"{filename}: {size_mb:.2f} MB")


========== Pilot ==========
best_kd_checkpoint.pt: 877.41 MB
latest_kd_checkpoint.pt: 2631.90 MB
pilot_results.json: 0.00 MB

========== ResumeTest ==========


In [ ]:
import os
import zipfile

BASE_DIR = "/content/drive/MyDrive/MiniBART_Research/E2"

for folder in ["Pilot", "ResumeTest"]:
    path = os.path.join(BASE_DIR, folder)

    print(f"\n========== {folder} ==========")

    if not os.path.exists(path):
        print("Folder does not exist.")
        continue

    files = os.listdir(path)

    if not files:
        print("Folder is empty.")
        continue

    for filename in files:
        filepath = os.path.join(path, filename)

        if not os.path.isfile(filepath):
            continue

        print(f"\nFile: {filename}")
        print(f"Size: {os.path.getsize(filepath) / (1024**2):.2f} MB")

        if filename.endswith(".pt"):
            try:
                with zipfile.ZipFile(filepath, "r") as z:
                    bad_file = z.testzip()

                    if bad_file is None:
                        print("ZIP integrity: PASSED")
                    else:
                        print("ZIP integrity: FAILED", bad_file)

                    print("Archive entries:", len(z.namelist()))

            except zipfile.BadZipFile:
                print("Not a ZIP-format PyTorch checkpoint.")


========== Pilot ==========

File: best_kd_checkpoint.pt
Size: 877.41 MB
ZIP integrity: PASSED
Archive entries: 266

File: latest_kd_checkpoint.pt
Size: 2631.90 MB
ZIP integrity: PASSED
Archive entries: 1046

File: pilot_results.json
Size: 0.00 MB

========== ResumeTest ==========
Folder is empty.


## Experiment Status — November 2026

- E2 implementation and local tests completed.
- E2 pilot training completed successfully (128 training examples, 32 validation examples, 1 epoch).
- Pilot global steps: 127.
- Best pilot validation loss: 4.14519.
- Best and latest pilot checkpoints saved to Google Drive and verified for ZIP integrity.
- Mid-epoch recovery test attempted but not conclusively completed.
- Colab GPU quota was exhausted, resulting in runtime disconnection.
- Full E2 training has not started.
- Final E2 evaluation has not been conducted.

The pilot checkpoints are temporary and may be deleted after their integrity verification and experiment records have been preserved.

**Next:** Resume full research experiments when GPU access is available.

# **CHANGES THAT I DID BUT WERE LOST COZ I DIDN'T SAVE IT: mounted drive, created a checkpoint and it survived runtime refresh... tested some other stuff**

# **ADDING LOST CELL BLOCKS**

# **DRIVE MOUNT AND GITHUB SYNC**

In [ ]:

import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Research")
PROJECT_DIR = REPO_DIR / "projects" / "01-mini-bart"

REPO_URL = "https://github.com/HeavenlyDemon27/Research.git"

def run_command(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

# Clone if missing; otherwise update the existing clone.
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Inspect it before removing or replacing anything."
        )

    run_command(["git", "clone", REPO_URL, str(REPO_DIR)])

else:
    run_command(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_command(["git", "checkout", "main"], cwd=REPO_DIR)
    run_command(
        ["git", "pull", "--ff-only", "origin", "main"],
        cwd=REPO_DIR
    )

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Mini-BART project directory not found: {PROJECT_DIR}"
    )

os.chdir(PROJECT_DIR)

print("\nRepository:", REPO_DIR)
print("Project directory:", Path.cwd())

print("\nCurrent Git commit:")
run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)

print("\nCurrent Git branch:")
run_command(["git", "branch", "--show-current"], cwd=REPO_DIR)


+ git fetch origin
+ git checkout main
+ git pull --ff-only origin main

Repository: /content/Research
Project directory: /content/Research/projects/01-mini-bart

Current Git commit:
+ git rev-parse HEAD

Current Git branch:
+ git branch --show-current


# **TESTING**

In [ ]:
import os
import sys
import subprocess

PROJECT_DIR = "/content/Research/projects/01-mini-bart"
os.chdir(PROJECT_DIR)

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "-v",
        "tests/test_kd_setup.py"
    ],
    check=False
)

print("Exit code:", result.returncode)

Exit code: 0


In [ ]:
import os
import sys
import subprocess

PROJECT_DIR = "/content/Research/projects/01-mini-bart"
os.chdir(PROJECT_DIR)

test_files = [
    "tests/test_distillation.py",
    "tests/test_kd_setup.py",
    "tests/test_train_kd.py",
    "tests/test_kd_checkpoint.py",
    "tests/test_kd_integration.py",
]

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", *test_files],
    check=False
)

print("\nFull KD test suite exit code:", result.returncode)


Full KD test suite exit code: 0


In [ ]:

import os
import sys
import importlib
import inspect
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/Research/projects/01-mini-bart")
REPO_DIR = Path("/content/Research")

os.chdir(PROJECT_DIR)

# Put the current repository first in Python's import path.
sys.path.insert(0, str(PROJECT_DIR))

# Clear any previously imported KD module.
sys.modules.pop("src.train_kd", None)

importlib.invalidate_caches()

# Import directly from the current repository.
train_kd = importlib.import_module("src.train_kd")

module_path = Path(train_kd.__file__).resolve()
expected_path = (PROJECT_DIR / "src" / "train_kd.py").resolve()

print("========== IMPORT VERIFICATION ==========")

print("Imported module:", train_kd.__name__)
print("Imported file:", module_path)
print("Expected file:", expected_path)

assert module_path == expected_path, (
    "ERROR: train_kd was imported from an unexpected location!"
)

print("\nGit commit:")
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    text=True
).strip()

print(commit)

print("\nGradient accumulation configuration:")
print(
    "KD_GRADIENT_ACCUMULATION_STEPS:",
    train_kd.KD_GRADIENT_ACCUMULATION_STEPS
)

assert train_kd.KD_GRADIENT_ACCUMULATION_STEPS == 8

print("\nTraining function signature:")
print(inspect.signature(train_kd.train_kd_one_epoch))

print("\nConfiguration:")
print("Epochs:", train_kd.KD_NUM_EPOCHS)

assert train_kd.KD_NUM_EPOCHS == 1

print("\nSUCCESS: Imported train_kd.py is from the updated repository.")


========== IMPORT VERIFICATION ==========
Imported module: src.train_kd
Imported file: /content/Research/projects/01-mini-bart/src/train_kd.py
Expected file: /content/Research/projects/01-mini-bart/src/train_kd.py

Git commit:
de6924c42989a8ba9094b287d14e89feb2ff7674

Gradient accumulation configuration:
KD_GRADIENT_ACCUMULATION_STEPS: 8

Training function signature:
(teacher, student, dataloader, optimizer, scheduler, device, scaler, global_step=0, temperature=2.0, lambda_kd=0.5, checkpoint_callback=None, checkpoint_every_steps=500, gradient_accumulation_steps=8)

Configuration:
Epochs: 1

SUCCESS: Imported train_kd.py is from the updated repository.


# **ACCUMULATION SMOKE TEST**

In [ ]:

import subprocess
import sys
import os

os.chdir("/content/Research/projects/01-mini-bart")

print("========== GRADIENT ACCUMULATION SMOKE TEST ==========")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        "tests/test_train_kd.py",
        "-k",
        "accumulation",
    ],
    check=True
)

print("\nGradient accumulation smoke test passed.")


# **CHECKPOINT AND RECOVERY SMOKE TEST**

In [ ]:

import subprocess
import sys
import os

os.chdir("/content/Research/projects/01-mini-bart")

print("========== CHECKPOINT RECOVERY SMOKE TEST ==========")

recovery_tests = [
    "tests/test_kd_checkpoint.py",
    "tests/test_kd_integration.py",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        *recovery_tests,
    ],
    check=True
)

print("\nCheckpoint and recovery smoke tests passed.")


In [ ]:
import os
import sys
import importlib
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Research")
PROJECT_DIR = REPO_DIR / "projects" / "01-mini-bart"

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Force a fresh import from the current repository
sys.modules.pop("src.train_kd", None)
importlib.invalidate_caches()

train_kd = importlib.import_module("src.train_kd")

actual_path = Path(train_kd.__file__).resolve()
expected_path = (PROJECT_DIR / "src" / "train_kd.py").resolve()

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    text=True
).strip()

print("Git commit:", commit)
print("Imported module:", actual_path)
print("Gradient accumulation:", train_kd.KD_GRADIENT_ACCUMULATION_STEPS)
print("Training epochs:", train_kd.KD_NUM_EPOCHS)

assert actual_path == expected_path, "Wrong train_kd.py imported!"
assert train_kd.KD_GRADIENT_ACCUMULATION_STEPS == 8
assert train_kd.KD_NUM_EPOCHS == 1

print("\nCode and configuration verification passed.")

Git commit: de6924c42989a8ba9094b287d14e89feb2ff7674
Imported module: /content/Research/projects/01-mini-bart/src/train_kd.py
Gradient accumulation: 8
Training epochs: 1

Code and configuration verification passed.


# **DRIVE BACKUP CHECKPOINT TEST**

In [ ]:

import os
import copy
import uuid
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from src.data import StatefulRandomSampler
from src.train import create_optimizer_and_scheduler
from src.train_kd import (
    get_kd_config,
    save_kd_checkpoint,
    load_kd_checkpoint,
)

# --------------------------------------------------
# 1. Confirm Google Drive is mounted
# --------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/AccumulationSmokeTest"
)

assert DRIVE_ROOT.is_dir(), (
    "Drive folder not found. Mount Google Drive first, "
    "then rerun this cell."
)

# Create a fresh directory. Never overwrite an earlier test.
TEST_DIR = DRIVE_ROOT / f"CheckpointRoundTrip_{uuid.uuid4().hex[:8]}"
TEST_DIR.mkdir(parents=True, exist_ok=False)

print("Test directory:", TEST_DIR)

# --------------------------------------------------
# 2. Tiny dummy model and dataset
# --------------------------------------------------

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(32, 16)
        self.projection = nn.Linear(16, 32)

    def forward(self, x):
        return self.projection(self.embedding(x))


class TinyDataset(Dataset):
    def __len__(self):
        return 8

    def __getitem__(self, index):
        return torch.tensor(
            [index + 1, index + 2, index + 3],
            dtype=torch.long
        )


torch.manual_seed(42)

student = TinyModel()

dataset = TinyDataset()
sampler = StatefulRandomSampler(
    dataset,
    generator=torch.Generator().manual_seed(42)
)

loader = DataLoader(
    dataset,
    batch_size=2,
    sampler=sampler,
    num_workers=0
)

optimizer, scheduler = create_optimizer_and_scheduler(
    student,
    total_training_steps=4
)

scaler = torch.amp.GradScaler("cpu", enabled=False)

config = get_kd_config(
    num_epochs=1,
    temperature=2.0,
    lambda_kd=0.5,
    gradient_accumulation_steps=8,
    checkpoint_every_steps=500,
    per_device_train_batch_size=2,
)

# --------------------------------------------------
# 3. Create meaningful optimizer/scheduler state
# --------------------------------------------------

optimizer.zero_grad(set_to_none=True)

dummy_loss = sum(
    parameter.square().mean()
    for parameter in student.parameters()
)

dummy_loss.backward()
optimizer.step()
scheduler.step()
optimizer.zero_grad(set_to_none=True)

# Move sampler to a meaningful mid-epoch position.
iterator = iter(loader)
next(iterator)

assert sampler.position == 2

# --------------------------------------------------
# 4. Save full recovery checkpoint to Google Drive
# --------------------------------------------------

checkpoint_path = save_kd_checkpoint(
    student=student,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_sampler=sampler,
    epoch=0,
    global_step=1,
    best_val_loss=3.25,
    output_dir=str(TEST_DIR),
    filename="drive_checkpoint_roundtrip.pt",
    config=config,
    include_training_state=True,
)

checkpoint_path = Path(checkpoint_path)

assert checkpoint_path.is_file()
assert str(checkpoint_path).startswith("/content/drive/")
assert checkpoint_path.stat().st_size > 0
assert not Path(str(checkpoint_path) + ".tmp").exists()

print("\nCheckpoint saved successfully.")
print("File size:", round(checkpoint_path.stat().st_size / (1024**2), 3), "MB")

# --------------------------------------------------
# 5. Capture expected state
# --------------------------------------------------

expected_weights = {
    name: parameter.detach().clone()
    for name, parameter in student.named_parameters()
}

expected_optimizer = copy.deepcopy(optimizer.state_dict())
expected_scheduler = copy.deepcopy(scheduler.state_dict())
expected_scaler = copy.deepcopy(scaler.state_dict())
expected_sampler = copy.deepcopy(sampler.state_dict())

# --------------------------------------------------
# 6. Deliberately corrupt current in-memory state
# --------------------------------------------------

with torch.no_grad():
    for parameter in student.parameters():
        parameter.add_(10.0)

# Advance sampler further.
next(iterator)

# --------------------------------------------------
# 7. Reload checkpoint from Google Drive
# --------------------------------------------------

restored = load_kd_checkpoint(
    checkpoint_path=str(checkpoint_path),
    student=student,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_sampler=sampler,
    device=torch.device("cpu"),
    expected_config=config,
)

# --------------------------------------------------
# 8. Verify restored state
# --------------------------------------------------

def assert_nested_equal(actual, expected):
    if isinstance(actual, torch.Tensor):
        assert torch.equal(actual, expected)
    elif isinstance(actual, dict):
        assert actual.keys() == expected.keys()
        for key in actual:
            assert_nested_equal(actual[key], expected[key])
    elif isinstance(actual, (list, tuple)):
        assert len(actual) == len(expected)
        for a, b in zip(actual, expected):
            assert_nested_equal(a, b)
    else:
        assert actual == expected

for name, parameter in student.named_parameters():
    assert torch.equal(parameter.detach(), expected_weights[name]), (
        f"Weight restoration failed: {name}"
    )

assert_nested_equal(optimizer.state_dict(), expected_optimizer)
assert_nested_equal(scheduler.state_dict(), expected_scheduler)
assert_nested_equal(scaler.state_dict(), expected_scaler)
assert_nested_equal(sampler.state_dict(), expected_sampler)

assert restored["epoch"] == 0
assert restored["global_step"] == 1
assert restored["best_val_loss"] == 3.25

# Independently reopen the file from its Drive path.
disk_checkpoint = torch.load(
    str(checkpoint_path),
    map_location="cpu",
    weights_only=False
)

required_keys = {
    "student_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
    "sampler_state_dict",
    "rng_state",
    "epoch",
    "global_step",
    "best_val_loss",
    "config",
}

assert required_keys.issubset(disk_checkpoint.keys())
assert disk_checkpoint["gradient_accumulation_steps"] == 8

print("\n" + "=" * 55)
print("DRIVE CHECKPOINT ROUND-TRIP: PASSED")
print("=" * 55)
print("Drive path:", checkpoint_path)
print("File size:", round(checkpoint_path.stat().st_size / (1024**2), 3), "MB")
print("Restored epoch:", restored["epoch"])
print("Restored global step:", restored["global_step"])
print("Restored sampler position:", sampler.position)
print("Model weights: verified")
print("Optimizer state: verified")
print("Scheduler state: verified")
print("Scaler state: verified")
print("Sampler state: verified")
print("RNG state: present in checkpoint")
print("Historical Pilot directory: untouched")


Test directory: /content/drive/MyDrive/MiniBART_Research/E2/AccumulationSmokeTest/CheckpointRoundTrip_ec3ca517
Checkpoint saved | drive_checkpoint_roundtrip.pt | next_epoch=0 | step=1

Checkpoint saved successfully.
File size: 0.036 MB
Resumed E2 | next_epoch=0 | global_step=1

DRIVE CHECKPOINT ROUND-TRIP: PASSED
Drive path: /content/drive/MyDrive/MiniBART_Research/E2/AccumulationSmokeTest/CheckpointRoundTrip_ec3ca517/drive_checkpoint_roundtrip.pt
File size: 0.036 MB
Restored epoch: 0
Restored global step: 1
Restored sampler position: 2
Model weights: verified
Optimizer state: verified
Scheduler state: verified
Scaler state: verified
Sampler state: verified
RNG state: present in checkpoint
Historical Pilot directory: untouched


# **RNG VERIFICATION FILE**

In [ ]:

import json
import random
from pathlib import Path

import numpy as np
import torch

TEST_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/"
    "AccumulationSmokeTest/CheckpointRoundTrip_4f69702f"
)

CHECKPOINT_PATH = TEST_DIR / "drive_checkpoint_roundtrip.pt"
EXPECTED_PATH = TEST_DIR / "expected_rng_after_restore.json"

assert CHECKPOINT_PATH.is_file(), "Checkpoint is missing from Drive."

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

rng_state = checkpoint["rng_state"]

# Generate expected values using independent RNG objects.
# This does not alter the current runtime's global RNG states.
python_rng = random.Random()
python_rng.setstate(rng_state["python"])
expected_python = python_rng.random()

numpy_rng = np.random.RandomState()
numpy_rng.set_state(rng_state["numpy"])
expected_numpy = float(numpy_rng.random())

torch_rng = torch.Generator(device="cpu")
torch_rng.set_state(rng_state["torch"])
expected_torch = torch.rand(5, generator=torch_rng).tolist()

expected = {
    "python": expected_python,
    "numpy": expected_numpy,
    "torch": expected_torch,
}

EXPECTED_PATH.write_text(json.dumps(expected, indent=2))

# Verify the sidecar can be reopened from Drive.
reloaded_expected = json.loads(EXPECTED_PATH.read_text())
assert reloaded_expected == expected

print("RNG verification reference saved successfully.")
print("Checkpoint:", CHECKPOINT_PATH)
print("Reference:", EXPECTED_PATH)
print("Expected Python random:", expected_python)
print("Expected NumPy random:", expected_numpy)
print("Expected Torch random:", expected_torch)


RNG verification reference saved successfully.
Checkpoint: /content/drive/MyDrive/MiniBART_Research/E2/AccumulationSmokeTest/CheckpointRoundTrip_4f69702f/drive_checkpoint_roundtrip.pt
Reference: /content/drive/MyDrive/MiniBART_Research/E2/AccumulationSmokeTest/CheckpointRoundTrip_4f69702f/expected_rng_after_restore.json
Expected Python random: 0.08111118093710878
Expected NumPy random: 0.9858553059755459
Expected Torch random: [0.3689272999763489, 0.07888573408126831, 0.10265827178955078, 0.792644739151001, 0.9277247190475464]


In [ ]:

import os
import sys
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

# --------------------------------------------------
# 1. Remount Google Drive
# --------------------------------------------------

drive.mount("/content/drive", force_remount=True)

TEST_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/"
    "AccumulationSmokeTest/CheckpointRoundTrip_4f69702f"
)

CHECKPOINT_PATH = TEST_DIR / "drive_checkpoint_roundtrip.pt"
EXPECTED_PATH = TEST_DIR / "expected_rng_after_restore.json"

assert CHECKPOINT_PATH.is_file(), "Original checkpoint not found."
assert EXPECTED_PATH.is_file(), "RNG reference file not found."

expected_rng = json.loads(EXPECTED_PATH.read_text())

# --------------------------------------------------
# 2. Reload the exact verified repository commit
# --------------------------------------------------

REPO_DIR = Path("/content/Research")
PROJECT_DIR = REPO_DIR / "projects" / "01-mini-bart"

REQUIRED_COMMIT = "de6924c42989a8ba9094b287d14e89feb2ff7674"
REPO_URL = "https://github.com/HeavenlyDemon27/Research.git"

if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Inspect it before proceeding."
        )
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    ["git", "checkout", "--detach", REQUIRED_COMMIT],
    cwd=REPO_DIR,
    check=True
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    text=True
).strip()

assert actual_commit == REQUIRED_COMMIT, (
    f"Wrong repository commit: {actual_commit}"
)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print("Repository commit verified:", actual_commit)

# --------------------------------------------------
# 3. Import the original E2 checkpoint functions
# --------------------------------------------------

from src.data import StatefulRandomSampler
from src.train import create_optimizer_and_scheduler
from src.train_kd import get_kd_config, load_kd_checkpoint

# --------------------------------------------------
# 4. Reconstruct the exact tiny model and dataset
# --------------------------------------------------

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(32, 16)
        self.projection = nn.Linear(16, 32)

    def forward(self, x):
        return self.projection(self.embedding(x))


class TinyDataset(Dataset):
    def __len__(self):
        return 8

    def __getitem__(self, index):
        return torch.tensor(
            [index + 1, index + 2, index + 3],
            dtype=torch.long
        )


torch.manual_seed(42)

student = TinyModel()
dataset = TinyDataset()

sampler = StatefulRandomSampler(
    dataset,
    generator=torch.Generator().manual_seed(42)
)

loader = DataLoader(
    dataset,
    batch_size=2,
    sampler=sampler,
    num_workers=0
)

optimizer, scheduler = create_optimizer_and_scheduler(
    student,
    total_training_steps=4
)

scaler = torch.amp.GradScaler("cpu", enabled=False)

config = get_kd_config(
    num_epochs=1,
    temperature=2.0,
    lambda_kd=0.5,
    gradient_accumulation_steps=8,
    checkpoint_every_steps=500,
    per_device_train_batch_size=2,
)

# --------------------------------------------------
# 5. Load checkpoint from Drive after runtime restart
# --------------------------------------------------

restored = load_kd_checkpoint(
    checkpoint_path=str(CHECKPOINT_PATH),
    student=student,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_sampler=sampler,
    device=torch.device("cpu"),
    expected_config=config,
)

# --------------------------------------------------
# 6. Verify checkpoint metadata and training state
# --------------------------------------------------

disk_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

assert restored["epoch"] == 0
assert restored["global_step"] == 1
assert restored["best_val_loss"] == 3.25
assert sampler.position == 2

# Verify model parameters against the saved checkpoint.
saved_weights = disk_checkpoint["student_state_dict"]

for name, parameter in student.named_parameters():
    assert torch.equal(
        parameter.detach().cpu(),
        saved_weights[name].cpu()
    ), f"Model weight mismatch: {name}"

# Verify optimizer, scheduler and scaler states.
assert optimizer.state_dict()["state"], (
    "Optimizer state was not restored."
)

assert scheduler.state_dict() == (
    disk_checkpoint["scheduler_state_dict"]
)

assert scaler.state_dict() == (
    disk_checkpoint["scaler_state_dict"]
)

assert sampler.state_dict()["position"] == (
    disk_checkpoint["sampler_state_dict"]["position"]
)

# --------------------------------------------------
# 7. Verify RNG restoration against pre-restart values
# --------------------------------------------------

actual_python = random.random()
actual_numpy = float(np.random.random())
actual_torch = torch.rand(5).tolist()

assert actual_python == expected_rng["python"], (
    "Python RNG restoration failed."
)

assert actual_numpy == expected_rng["numpy"], (
    "NumPy RNG restoration failed."
)

assert np.allclose(
    actual_torch,
    expected_rng["torch"],
    rtol=0,
    atol=0
), "Torch RNG restoration failed."

# --------------------------------------------------
# 8. Final report
# --------------------------------------------------

print("\n" + "=" * 58)
print("COLD-RESTART CHECKPOINT RECOVERY: PASSED")
print("=" * 58)

print("Repository commit:", actual_commit)
print("Checkpoint:", CHECKPOINT_PATH)
print("Restored epoch:", restored["epoch"])
print("Restored global step:", restored["global_step"])
print("Restored sampler position:", sampler.position)
print("Model weights: verified")
print("Optimizer state: verified")
print("Scheduler state: verified")
print("Scaler state: verified")
print("Sampler state: verified")
print("Python RNG: verified")
print("NumPy RNG: verified")
print("Torch RNG: verified")
print("\nOriginal Drive checkpoint preserved.")


Mounted at /content/drive
Repository commit verified: de6924c42989a8ba9094b287d14e89feb2ff7674
Resumed E2 | next_epoch=0 | global_step=1

COLD-RESTART CHECKPOINT RECOVERY: PASSED
Repository commit: de6924c42989a8ba9094b287d14e89feb2ff7674
Checkpoint: /content/drive/MyDrive/MiniBART_Research/E2/AccumulationSmokeTest/CheckpointRoundTrip_4f69702f/drive_checkpoint_roundtrip.pt
Restored epoch: 0
Restored global step: 1
Restored sampler position: 2
Model weights: verified
Optimizer state: verified
Scheduler state: verified
Scaler state: verified
Sampler state: verified
Python RNG: verified
NumPy RNG: verified
Torch RNG: verified

Original Drive checkpoint preserved.


# **FINAL TESTING**

In [3]:

import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HeavenlyDemon27/Research.git"
REPO_DIR = Path("/content/Research")
PROJECT_DIR = REPO_DIR / "projects" / "01-mini-bart"

VERIFIED_COMMIT = "de6924c42989a8ba9094b287d14e89feb2ff7674"

def run_git(args):
    print("+ git", " ".join(args))
    subprocess.run(
        ["git", "-C", str(REPO_DIR), *args],
        check=True
    )

if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Inspect it before proceeding."
        )
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    run_git(["fetch", "origin"])

# Do not silently discard uncommitted changes.
status = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
    text=True
).strip()

if status:
    raise RuntimeError(
        "Repository has uncommitted changes. Inspect them before checkout:\n"
        + status
    )

run_git(["checkout", "--detach", VERIFIED_COMMIT])

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == VERIFIED_COMMIT
assert PROJECT_DIR.is_dir()

os.chdir(PROJECT_DIR)

required_files = [
    "src/data.py",
    "src/model.py",
    "src/train.py",
    "src/train_kd.py",
    "src/distillation.py",
    "tests/test_kd_checkpoint.py",
    "tests/test_kd_integration.py",
]

for filename in required_files:
    assert (PROJECT_DIR / filename).is_file(), (
        f"Missing required file: {filename}"
    )

print("\nRepository setup passed.")
print("Project:", PROJECT_DIR)
print("Commit:", actual_commit)


+ git checkout --detach de6924c42989a8ba9094b287d14e89feb2ff7674

Repository setup passed.
Project: /content/Research/projects/01-mini-bart
Commit: de6924c42989a8ba9094b287d14e89feb2ff7674


In [4]:

import os
import torch
from datasets import load_dataset

from src.data import (
    load_bart_tokenizer,
    prepare_dataset,
    create_dataloaders,
)

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME = "facebook/bart-large-cnn"
DATASET_NAME = "abisee/cnn_dailymail"
DATASET_VERSION = "3.0.0"

PILOT_TRAIN_SAMPLES = 128
PILOT_VAL_SAMPLES = 32
BATCH_SIZE = 2

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

print("Loading tokenizer...")
tokenizer = load_bart_tokenizer(MODEL_NAME)

print("Loading pilot training subset...")
train_raw = load_dataset(
    DATASET_NAME,
    DATASET_VERSION,
    split=f"train[:{PILOT_TRAIN_SAMPLES}]",
)

print("Loading pilot validation subset...")
val_raw = load_dataset(
    DATASET_NAME,
    DATASET_VERSION,
    split=f"validation[:{PILOT_VAL_SAMPLES}]",
)

assert len(train_raw) == PILOT_TRAIN_SAMPLES
assert len(val_raw) == PILOT_VAL_SAMPLES

raw_dataset = {
    "train": train_raw,
    "validation": val_raw,
}

print("Tokenizing pilot data...")
tokenized_dataset = prepare_dataset(
    raw_dataset,
    tokenizer,
)

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

EXPECTED_MICROBATCHES = 64
EXPECTED_UPDATES = 8

assert len(train_loader) == EXPECTED_MICROBATCHES
assert len(val_loader) == 16
assert len(test_loader) == 0

assert hasattr(train_loader.sampler, "state_dict")

print("\nPilot data preparation passed.")
print("Training examples:", len(train_raw))
print("Validation examples:", len(val_raw))
print("Training microbatches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Expected optimizer updates:", EXPECTED_UPDATES)
print("Test loader remains empty.")


Loading tokenizer...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loading pilot training subset...


... (14 outputs left out)

Loading pilot validation subset...
Tokenizing pilot data...


Tokenizing train split:   0%|          | 0/128 [00:00<?, ? examples/s]

Tokenizing validation split:   0%|          | 0/32 [00:00<?, ? examples/s]


Pilot data preparation passed.
Training examples: 128
Validation examples: 32
Training microbatches: 64
Validation batches: 16
Expected optimizer updates: 8
Test loader remains empty.


In [10]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research"
)

E2_DIR = DRIVE_PROJECT_DIR / "E2"
PILOT_DIR = E2_DIR / "Pilot"
FINAL_DIR = E2_DIR / "Final"

PILOT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

assert E2_DIR.is_dir()

print("Drive mounted successfully.")
print("E2 directory:", E2_DIR)
print("Pilot directory:", PILOT_DIR)
print("Final directory:", FINAL_DIR)

Mounted at /content/drive
Drive mounted successfully.
E2 directory: /content/drive/MyDrive/MiniBART_Research/E2
Pilot directory: /content/drive/MyDrive/MiniBART_Research/E2/Pilot
Final directory: /content/drive/MyDrive/MiniBART_Research/E2/Final


In [11]:
import torch
from collections import Counter

assert torch.cuda.is_available(), "CUDA is unavailable!"

device = torch.device("cuda:0")

print("Moving teacher to:", device)
teacher_model = teacher_model.to(device)

print("Moving student to:", device)
student_model = student_model.to(device)

teacher_model.eval()
student_model.train()

print("\n========== DEVICE VERIFICATION ==========")

for name, model in [
    ("Teacher", teacher_model),
    ("Student", student_model),
]:
    parameter_devices = Counter(
        str(p.device) for p in model.parameters()
    )

    buffer_devices = Counter(
        str(b.device) for b in model.buffers()
    )

    print(f"\n{name}")
    print("Parameter devices:", parameter_devices)
    print("Buffer devices:", buffer_devices)
    print("Embedding device:", model.model.shared.weight.device)

    assert all(
        p.device.type == "cuda"
        for p in model.parameters()
    ), f"{name} still has CPU parameters!"

    assert all(
        b.device.type == "cuda"
        for b in model.buffers()
    ), f"{name} still has CPU buffers!"

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("\nSUCCESS: Both models are on the T4.")

Moving teacher to: cuda:0
Moving student to: cuda:0

========== DEVICE VERIFICATION ==========

Teacher
Parameter devices: Counter({'cuda:0': 511})
Buffer devices: Counter({'cuda:0': 1})
Embedding device: cuda:0

Student
Parameter devices: Counter({'cuda:0': 259})
Buffer devices: Counter({'cuda:0': 1})
Embedding device: cuda:0

GPU memory allocated: 2.37 GB

SUCCESS: Both models are on the T4.


In [5]:

import torch

from src.model import (
    load_teacher_model,
    prepare_kd_models,
)

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Select a GPU runtime before proceeding."
)

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Total GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB",
)

torch.manual_seed(42)

print("\nLoading pretrained BART teacher...")
teacher_model = load_teacher_model(MODEL_NAME)

print("Preparing fresh Mini-BART student...")
teacher_model, student_model = prepare_kd_models(teacher_model)

assert len(student_model.model.encoder.layers) == 6
assert len(student_model.model.decoder.layers) == 6

assert all(
    not parameter.requires_grad
    for parameter in teacher_model.parameters()
)

assert all(
    parameter.requires_grad
    for parameter in student_model.parameters()
)

print("\nModel preparation passed.")
print("Teacher parameters:", sum(
    p.numel() for p in teacher_model.parameters()
))
print("Student parameters:", sum(
    p.numel() for p in student_model.parameters()
))
print("Teacher frozen: True")
print("Student trainable: True")


GPU: Tesla T4
Total GPU memory: 14.56 GB

Loading pretrained BART teacher...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Preparing fresh Mini-BART student...
Student architecture verified: 6E + 6D
Embedding/LM-head weight tying verified.

Model preparation passed.
Teacher parameters: 406290432
Student parameters: 229933056
Teacher frozen: True
Student trainable: True


In [15]:
sampler = train_loader.sampler

print("Sampler position:", sampler.position)
print("Sampler epoch:", sampler.epoch)
print("Permutation initialized:", sampler.permutation is not None)
print("Remaining samples:", len(sampler))

Sampler position: 4
Sampler epoch: 0
Permutation initialized: True
Remaining samples: 124


In [16]:
assert sampler.position == 4
assert sampler.epoch == 0

# Restore a fresh, deterministic training permutation.
sampler.permutation = None
sampler.position = 0
sampler.epoch = 0
sampler.generator.manual_seed(42)

print("Restored training sampler.")
print("Sampler length:", len(sampler))
print("Training microbatches:", len(train_loader))

assert len(sampler) == 128
assert len(train_loader) == 64

Restored training sampler.
Sampler length: 128
Training microbatches: 64


In [12]:

import torch
from src.distillation import combined_distillation_loss

teacher_model.eval()
student_model.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

batch = next(iter(train_loader))
batch = {
    key: value.to(device, non_blocking=True)
    for key, value in batch.items()
}

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        teacher_outputs = teacher_model(**batch)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    student_outputs = student_model(**batch)

    losses = combined_distillation_loss(
        teacher_logits=teacher_outputs.logits,
        student_logits=student_outputs.logits,
        labels=batch["labels"],
        temperature=2.0,
        lambda_kd=0.5,
    )

assert torch.isfinite(losses["loss"]).item()
assert torch.isfinite(losses["ce_loss"]).item()
assert torch.isfinite(losses["kd_loss"]).item()

losses["loss"].backward()

student_has_gradients = any(
    parameter.grad is not None
    for parameter in student_model.parameters()
)

assert student_has_gradients, "Student received no gradients."

student_model.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print("GPU teacher/student forward-backward test PASSED.")
print("Total loss:", round(losses["loss"].item(), 4))
print("CE loss:", round(losses["ce_loss"].item(), 4))
print("KD loss:", round(losses["kd_loss"].item(), 4))
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)


GPU teacher/student forward-backward test PASSED.
Total loss: 7.8793
CE loss: 7.1404
KD loss: 1.4778
Peak GPU memory: 3.64 GB


In [17]:
from pathlib import Path

PILOT_RUN_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/"
    "Pilot_Accumulation8_20261124T121319Z_91bc7a"
)

sampler = train_loader.sampler

print("Pilot directory exists:", PILOT_RUN_DIR.is_dir())
print("Existing directory contents:")

if PILOT_RUN_DIR.is_dir():
    for item in sorted(PILOT_RUN_DIR.iterdir()):
        print(" ", item.name, "-", "directory" if item.is_dir() else "file")

print("\nSampler position:", sampler.position)
print("Sampler epoch:", sampler.epoch)
print("Sampler permutation initialized:", sampler.permutation is not None)
print("Remaining samples:", len(sampler))
print("Training microbatches:", len(train_loader))
print("Training batch size:", train_loader.batch_size)

Pilot directory exists: True
Existing directory contents:

Sampler position: 0
Sampler epoch: 0
Sampler permutation initialized: False
Remaining samples: 128
Training microbatches: 64
Training batch size: 2


In [18]:

import json
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch

from src.train_kd import (
    train_kd_model,
    get_kd_config,
)

# --------------------------------------------------
# 1. Reuse the existing, isolated experiment directory
# --------------------------------------------------

assert "PILOT_RUN_DIR" in globals(), (
    "Existing pilot directory variable is missing."
)

assert PILOT_RUN_DIR.is_dir(), (
    "Existing pilot directory does not exist."
)

assert not any(PILOT_RUN_DIR.iterdir()), (
    "Pilot directory is not empty. Do not overwrite its contents."
)

RUN_ID = PILOT_RUN_DIR.name.removeprefix(
    "Pilot_Accumulation8_"
)

print("Reusing existing pilot directory:")
print(PILOT_RUN_DIR)
print("Run ID:", RUN_ID)
# --------------------------------------------------
# 2. Explicit experiment configuration
# --------------------------------------------------

NUM_EPOCHS = 1
TEMPERATURE = 2.0
LAMBDA_KD = 0.5
GRADIENT_ACCUMULATION_STEPS = 8
CHECKPOINT_EVERY_STEPS = 2

EXPECTED_MICROBATCHES = 64
EXPECTED_OPTIMIZER_UPDATES = 8

assert len(train_loader) == EXPECTED_MICROBATCHES
assert train_loader.batch_size == BATCH_SIZE
assert train_loader.num_workers == 0

pilot_config = get_kd_config(
    num_epochs=NUM_EPOCHS,
    temperature=TEMPERATURE,
    lambda_kd=LAMBDA_KD,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
)

# --------------------------------------------------
# 3. Record hardware and launch training
# --------------------------------------------------
# Verify clean training state
sampler = train_loader.sampler

assert sampler.position == 0
assert sampler.epoch == 0
assert sampler.permutation is None

assert len(train_loader) == EXPECTED_MICROBATCHES
assert len(train_loader) == 64
assert train_loader.batch_size == 2
assert train_loader.num_workers == 0

assert GRADIENT_ACCUMULATION_STEPS == 8

print("Pre-training checks PASSED.")
print("Expected optimizer updates:", EXPECTED_OPTIMIZER_UPDATES)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

print("\nStarting E2 accumulation-aware GPU pilot...")
print("Expected microbatches:", EXPECTED_MICROBATCHES)
print("Expected optimizer updates:", EXPECTED_OPTIMIZER_UPDATES)
print("Checkpoint cadence:", CHECKPOINT_EVERY_STEPS)

pilot_results = train_kd_model(
    teacher=teacher_model,
    student=student_model,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    device=device,
    output_dir=str(PILOT_RUN_DIR),
    num_epochs=NUM_EPOCHS,
    temperature=TEMPERATURE,
    lambda_kd=LAMBDA_KD,
    resume_from=None,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)

elapsed_seconds = time.perf_counter() - start_time
peak_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

# --------------------------------------------------
# 4. Save experiment record
# --------------------------------------------------

pilot_record = {
    "experiment": "E2 controlled GPU pilot",
    "run_id": RUN_ID,
    "repository_commit": "de6924c42989a8ba9094b287d14e89feb2ff7674",
    "teacher": MODEL_NAME,
    "dataset": DATASET_NAME,
    "dataset_version": DATASET_VERSION,
    "train_examples": PILOT_TRAIN_SAMPLES,
    "validation_examples": PILOT_VAL_SAMPLES,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "microbatches_per_epoch": EXPECTED_MICROBATCHES,
    "expected_optimizer_updates": EXPECTED_OPTIMIZER_UPDATES,
    "actual_optimizer_updates": pilot_results["global_step"],
    "num_epochs": NUM_EPOCHS,
    "temperature": TEMPERATURE,
    "lambda_kd": LAMBDA_KD,
    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "training_seconds": elapsed_seconds,
    "peak_gpu_memory_gb": peak_memory_gb,
    "best_val_loss": pilot_results["best_val_loss"],
    "history": pilot_results["history"],
    "config": pilot_config,
}

record_path = PILOT_RUN_DIR / "pilot_results.json"

with open(record_path, "w", encoding="utf-8") as f:
    json.dump(pilot_record, f, indent=2)

print("\nPilot training finished.")
print("Training time:", round(elapsed_seconds / 60, 2), "minutes")
print("Peak allocated GPU memory:", round(peak_memory_gb, 2), "GB")
print("Successful optimizer updates:", pilot_results["global_step"])
print("Best validation loss:", pilot_results["best_val_loss"])
print("Results record:", record_path)


Reusing existing pilot directory:
/content/drive/MyDrive/MiniBART_Research/E2/Pilot_Accumulation8_20261124T121319Z_91bc7a
Run ID: 20261124T121319Z_91bc7a
Pre-training checks PASSED.
Expected optimizer updates: 8

Starting E2 accumulation-aware GPU pilot...
Expected microbatches: 64
Expected optimizer updates: 8
Checkpoint cadence: 2

========== E2 Epoch 1/1 ==========


E2 Training:   0%|          | 0/64 [00:00<?, ?it/s]

Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=2
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=4
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=0 | step=6


E2 Validation:   0%|          | 0/16 [00:00<?, ?it/s]


Epoch 1 Results
Train Total: 6.5189 | CE: 5.8046 | KD: 1.4285
Val Total: 5.1623 | CE: 4.5592 | KD: 1.2062
Checkpoint saved | best_kd_checkpoint.pt | next_epoch=1 | step=8
Checkpoint saved | latest_kd_checkpoint.pt | next_epoch=1 | step=8

Pilot training finished.
Training time: 2.86 minutes
Peak allocated GPU memory: 6.12 GB
Successful optimizer updates: 8
Best validation loss: 5.162307441234589
Results record: /content/drive/MyDrive/MiniBART_Research/E2/Pilot_Accumulation8_20261124T121319Z_91bc7a/pilot_results.json


In [19]:
import json
from pathlib import Path

PILOT_RUN_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/"
    "Pilot_Accumulation8_20261124T121319Z_91bc7a"
)

print("PILOT ARTIFACTS")
for path in sorted(PILOT_RUN_DIR.iterdir()):
    print(f"{path.name:35s} {path.stat().st_size:,} bytes")

results_path = PILOT_RUN_DIR / "pilot_results.json"
assert results_path.is_file(), "Missing pilot_results.json"

with open(results_path, "r") as f:
    results = json.load(f)

print("\nPILOT RESULTS RECORD")
print(json.dumps(results, indent=2))

assert results_path.stat().st_size > 0

PILOT ARTIFACTS
best_kd_checkpoint.pt               920,036,419 bytes
latest_kd_checkpoint.pt             2,759,751,591 bytes
pilot_results.json                  1,712 bytes

PILOT RESULTS RECORD
{
  "experiment": "E2 controlled GPU pilot",
  "run_id": "20261124T121319Z_91bc7a",
  "repository_commit": "de6924c42989a8ba9094b287d14e89feb2ff7674",
  "teacher": "facebook/bart-large-cnn",
  "dataset": "abisee/cnn_dailymail",
  "dataset_version": "3.0.0",
  "train_examples": 128,
  "validation_examples": 32,
  "batch_size": 2,
  "gradient_accumulation_steps": 8,
  "microbatches_per_epoch": 64,
  "expected_optimizer_updates": 8,
  "actual_optimizer_updates": 8,
  "num_epochs": 1,
  "temperature": 2.0,
  "lambda_kd": 0.5,
  "checkpoint_every_steps": 2,
  "training_seconds": 171.6659779050001,
  "peak_gpu_memory_gb": 6.115874767303467,
  "best_val_loss": 5.162307441234589,
  "history": [
    {
      "epoch": 1,
      "train": {
        "loss": 6.518880568444729,
        "ce_loss": 5.80462506413

TESTING IS COMPLETED SUCCESSFULLY

# **E2 PREPROCESSING**

In [20]:

import os
import sys
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ------------------------------------------------------------
# 1. VERIFY PROJECT LOCATION AND CODE VERSION
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/Research/projects/01-mini-bart")

assert PROJECT_DIR.exists(), "Mini-BART project directory not found."

os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import subprocess

EXPECTED_COMMIT = "de6924c42989a8ba9094b287d14e89feb2ff7674"

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_DIR,
    text=True,
).strip()

assert actual_commit == EXPECTED_COMMIT, (
    f"Wrong repository commit!\n"
    f"Expected: {EXPECTED_COMMIT}\n"
    f"Actual: {actual_commit}"
)

print("Verified Git commit:", actual_commit)

# ------------------------------------------------------------
# 2. IMPORT DATA FUNCTIONS
# ------------------------------------------------------------

from src.data import (
    load_bart_tokenizer,
    load_cnn_dailymail,
    prepare_dataset,
    create_dataloaders,
)

MODEL_NAME = "facebook/bart-large-cnn"

BATCH_SIZE = 1
NUM_WORKERS = 0

EXPECTED_TRAIN_EXAMPLES = 287113
EXPECTED_VAL_EXAMPLES = 13368

# ------------------------------------------------------------
# 3. LOAD TOKENIZER AND FULL DATASET
# ------------------------------------------------------------

print("\nLoading BART tokenizer...")

tokenizer = load_bart_tokenizer(MODEL_NAME)

print("\nLoading FULL CNN/DailyMail 3.0.0 dataset...")

raw_dataset = load_cnn_dailymail()

assert len(raw_dataset["train"]) == EXPECTED_TRAIN_EXAMPLES
assert len(raw_dataset["validation"]) == EXPECTED_VAL_EXAMPLES

print("\nRaw dataset verified.")
print("Training examples:", len(raw_dataset["train"]))
print("Validation examples:", len(raw_dataset["validation"]))

# ------------------------------------------------------------
# 4. TOKENIZE FULL TRAINING AND VALIDATION DATA
# ------------------------------------------------------------

print("\nTokenizing FULL training and validation datasets...")

tokenized_dataset = prepare_dataset(
    dataset=raw_dataset,
    tokenizer=tokenizer,
)

assert len(tokenized_dataset["train"]) == EXPECTED_TRAIN_EXAMPLES
assert len(tokenized_dataset["validation"]) == EXPECTED_VAL_EXAMPLES
assert len(tokenized_dataset["test"]) == 0

# ------------------------------------------------------------
# 5. CREATE FULL-RUN DATALOADERS
# ------------------------------------------------------------

print("\nCreating full-run DataLoaders...")

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

# ------------------------------------------------------------
# 6. VERIFY EVERYTHING
# ------------------------------------------------------------

assert len(train_loader) == EXPECTED_TRAIN_EXAMPLES
assert len(val_loader) == EXPECTED_VAL_EXAMPLES
assert len(test_loader) == 0

assert train_loader.batch_size == 1
assert train_loader.num_workers == 0
assert val_loader.batch_size == 1

assert hasattr(train_loader.sampler, "state_dict")

print("\n========== FULL E2 DATA PREPARATION PASSED ==========")

print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))
print("Test examples:", len(tokenized_dataset["test"]))

print("\nTraining microbatches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("Per-device batch size:", BATCH_SIZE)
print("Gradient accumulation: 8")
print("Effective batch size:", BATCH_SIZE * 8)

print("Sampler:", type(train_loader.sampler).__name__)
print("DataLoader workers:", train_loader.num_workers)

print("\nTest split remains untouched.")
print("=====================================================")


Verified Git commit: de6924c42989a8ba9094b287d14e89feb2ff7674

Loading BART tokenizer...

Loading FULL CNN/DailyMail 3.0.0 dataset...

Raw dataset verified.
Training examples: 287113
Validation examples: 13368

Tokenizing FULL training and validation datasets...


Tokenizing train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Tokenizing validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]


Creating full-run DataLoaders...

========== FULL E2 DATA PREPARATION PASSED ==========
Training examples: 287113
Validation examples: 13368
Test examples: 0

Training microbatches: 287113
Validation batches: 13368
Per-device batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Sampler: StatefulRandomSampler
DataLoader workers: 0

Test split remains untouched.


In [21]:
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoTokenizer
import json

# Drive must already be mounted
TOKENIZED_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research/E2/SharedTokenizedData/"
    "mini_bart_tokenized_512_128_v1"
)

assert TOKENIZED_DIR.exists(), f"Dataset folder not found: {TOKENIZED_DIR}"

print("Folder contents:")
print(sorted(p.name for p in TOKENIZED_DIR.iterdir())[:30])

# Load the saved Hugging Face DatasetDict
tokenized_dataset = load_from_disk(str(TOKENIZED_DIR))

print("\nLoaded object:", type(tokenized_dataset))
print("Available splits:", list(tokenized_dataset.keys()))

# Load the expected tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

# Inspect each split
for split_name, dataset in tokenized_dataset.items():
    print(f"\n{'=' * 55}")
    print(f"SPLIT: {split_name}")
    print("Examples:", len(dataset))
    print("Columns:", dataset.column_names)
    print("Features:", dataset.features)

    required = {"input_ids", "attention_mask", "labels"}
    missing = required - set(dataset.column_names)
    print("Missing required columns:", missing)

    if missing:
        continue

    # Scan every example in batches to find actual maximum lengths
    max_input_len = 0
    max_target_len = 0
    label_ignore_count = 0

    for start in range(0, len(dataset), 2048):
        batch = dataset[start : start + 2048]

        max_input_len = max(
            max_input_len,
            max(len(x) for x in batch["input_ids"])
        )
        max_target_len = max(
            max_target_len,
            max(len(x) for x in batch["labels"])
        )
        label_ignore_count += sum(
            token_id == -100
            for row in batch["labels"]
            for token_id in row
        )

    print("Maximum source token length:", max_input_len)
    print("Maximum target token length:", max_target_len)
    print("Number of -100 label values:", label_ignore_count)

    # Decode one example for a sanity check
    sample = dataset[0]
    source_text = tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True
    )
    target_ids = [x for x in sample["labels"] if x != -100]
    target_text = tokenizer.decode(
        target_ids,
        skip_special_tokens=True
    )

    print("\nDecoded source sample:")
    print(source_text[:700])

    print("\nDecoded target sample:")
    print(target_text[:400])

Folder contents:
['dataset_dict.json', 'train', 'validation']

Loaded object: <class 'datasets.dataset_dict.DatasetDict'>
Available splits: ['train', 'validation']

SPLIT: train
Examples: 287113
Columns: ['input_ids', 'attention_mask', 'labels']
Features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}
Missing required columns: set()
Maximum source token length: 512
Maximum target token length: 128
Number of -100 label values: 0

Decoded source sample:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those 

In [22]:
from src.data import load_bart_tokenizer, create_dataloaders

MODEL_NAME = "facebook/bart-large-cnn"

# Load the expected BART tokenizer
tokenizer = load_bart_tokenizer(MODEL_NAME)

# Create E2 DataLoaders from the saved tokenized artifact
BATCH_SIZE = 1

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

# Verify loader configuration
assert len(train_loader.dataset) == 287113
assert len(val_loader.dataset) == 13368
assert train_loader.batch_size == 1
assert train_loader.num_workers == 0
assert hasattr(train_loader.sampler, "state_dict")

print("E2 DataLoaders created successfully.")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("Training batch size:", train_loader.batch_size)
print("Sampler:", type(train_loader.sampler).__name__)

E2 DataLoaders created successfully.
Training batches: 287113
Validation batches: 13368
Test batches: 0
Training batch size: 1
Sampler: StatefulRandomSampler




> **one final dataloader check**



In [ ]:
# Inspect one example through the DataLoader's collator
sample = tokenized_dataset["train"][0]
batch = train_loader.collate_fn([sample])

print("Batch keys:", batch.keys())

for key, value in batch.items():
    if hasattr(value, "shape"):
        print(
            f"{key}: shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )
    else:
        print(f"{key}: {type(value)}")

print("\nLabels:")
print("Minimum:", batch["labels"].min().item())
print("Maximum:", batch["labels"].max().item())
print("Contains -100:", (batch["labels"] == -100).any().item())

to directly load dataset to avoid preprocessing again brehhh :

In [ ]:
from datasets import load_from_disk

TOKENIZED_DIR = (
    "/content/drive/MyDrive/MiniBART_Research/"
    "E2/SharedTokenizedData/"
    "mini_bart_tokenized_512_128_v1"
)

tokenized_dataset = load_from_disk(TOKENIZED_DIR)



# **CHECKING FINAL CONFIG**



In [ ]:

import inspect
import src.model as model_module
import src.train_kd as train_kd_module
import src.train as train_module

print("=" * 65)
print("1. MODEL MODULE — AVAILABLE CLASSES AND FUNCTIONS")
print("=" * 65)

for name, obj in inspect.getmembers(model_module):
    if inspect.isfunction(obj) or inspect.isclass(obj):
        if obj.__module__ == model_module.__name__:
            try:
                print(f"\n{name}{inspect.signature(obj)}")
            except (TypeError, ValueError):
                print(f"\n{name}")

print("\n" + "=" * 65)
print("2. MODEL INITIALIZATION FUNCTIONS — SOURCE")
print("=" * 65)

for name, obj in inspect.getmembers(model_module, inspect.isfunction):
    if obj.__module__ == model_module.__name__:
        print(f"\n--- {name} ---")
        print(inspect.getsource(obj))

print("\n" + "=" * 65)
print("3. E2 TRAINING FUNCTION SIGNATURE")
print("=" * 65)

print(inspect.signature(train_kd_module.train_kd_model))

print("\n" + "=" * 65)
print("4. OPTIMIZER FUNCTION SIGNATURE")
print("=" * 65)

print(inspect.signature(train_module.create_optimizer_and_scheduler))

print("\n" + "=" * 65)
print("5. E2 DEFAULT CONFIGURATION")
print("=" * 65)

print("Temperature:", train_kd_module.KD_TEMPERATURE)
print("KD lambda:", train_kd_module.KD_LAMBDA)
print("Default epochs:", train_kd_module.KD_NUM_EPOCHS)
print("Checkpoint interval:", train_kd_module.KD_CHECKPOINT_EVERY_STEPS)

print("\nPreflight inspection completed.")


In [25]:

import torch
from src.model import load_teacher_model, prepare_kd_models

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load pretrained teacher
teacher_model = load_teacher_model("facebook/bart-large-cnn")

# Freeze teacher and create a fresh 6E + 6D student
teacher_model, student_model = prepare_kd_models(teacher_model)

# Move both models to the selected device
teacher_model = teacher_model.to(device)
student_model = student_model.to(device)

# Verify model properties
print("\nTeacher encoder layers:", len(teacher_model.model.encoder.layers))
print("Teacher decoder layers:", len(teacher_model.model.decoder.layers))

print("Student encoder layers:", len(student_model.model.encoder.layers))
print("Student decoder layers:", len(student_model.model.decoder.layers))

print("Teacher trainable parameters:",
      sum(p.numel() for p in teacher_model.parameters() if p.requires_grad))

print("Student trainable parameters:",
      sum(p.numel() for p in student_model.parameters() if p.requires_grad))

print("Teacher training mode:", teacher_model.training)
print("Student training mode:", student_model.training)

assert not any(p.requires_grad for p in teacher_model.parameters())
assert all(p.requires_grad for p in student_model.parameters())
assert len(student_model.model.encoder.layers) == 6
assert len(student_model.model.decoder.layers) == 6

print("\nE2 model initialization successful.")


Device: cuda


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Student architecture verified: 6E + 6D
Embedding/LM-head weight tying verified.

Teacher encoder layers: 12
Teacher decoder layers: 12
Student encoder layers: 6
Student decoder layers: 6
Teacher trainable parameters: 0
Student trainable parameters: 229933056
Teacher training mode: False
Student training mode: True

E2 model initialization successful.


In [ ]:

import inspect
import src.train_kd as train_kd_module
import src.train as train_module

print("=" * 60)
print("E2 TRAINING FUNCTION")
print("=" * 60)
print(inspect.getsource(train_kd_module.train_kd_model))

print("\n" + "=" * 60)
print("OPTIMIZER AND SCHEDULER")
print("=" * 60)
print(inspect.getsource(train_module.create_optimizer_and_scheduler))


In [ ]:
import inspect
import src.train_kd as train_kd_module
import src.distillation as distillation_module

print("========== TRAINING LOOP ==========")
print(inspect.getsource(train_kd_module.train_kd_one_epoch))

print("\n========== KD LOSS ==========")
print(inspect.getsource(distillation_module))

print("\n========== KD CONFIGURATION ==========")
print(inspect.getsource(train_kd_module.get_kd_config))

print("\n========== TRAINING CONSTANTS ==========")
for name in [
    "LEARNING_RATE",
    "WEIGHT_DECAY",
    "WARMUP_RATIO",
    "KD_NUM_EPOCHS",
    "KD_TEMPERATURE",
    "KD_LAMBDA",
    "KD_CHECKPOINT_EVERY_STEPS",
    "KD_GRADIENT_ACCUMULATION_STEPS",
]:
    print(name, "=", getattr(train_kd_module, name, "NOT FOUND"))

In [ ]:
import inspect
import src.train_kd as train_kd_module
from src.data import StatefulRandomSampler

print("========== SAVE CHECKPOINT ==========")
print(inspect.getsource(train_kd_module.save_kd_checkpoint))

print("\n========== LOAD CHECKPOINT ==========")
print(inspect.getsource(train_kd_module.load_kd_checkpoint))

print("\n========== SAMPLER STATE ==========")
print(inspect.getsource(StatefulRandomSampler.state_dict))

# **TRAINING E2**

In [ ]:

# ============================================================
# E2 — FULL KNOWLEDGE DISTILLATION TRAINING
# Final launch / automatic recovery cell
#
# Fresh run:
#   Creates a new teacher-initialized 6E+6D student.
#
# Resume:
#   Restores the latest validated recovery checkpoint.
#
# Already completed:
#   Detects epoch == NUM_EPOCHS and does not train again.
#
# Pilot and ResumeTest artifacts are never touched.
# ============================================================


# ============================================================
# 1. IMPORTS AND ENVIRONMENT
# ============================================================

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import sys
import gc
import json
import random
import subprocess

from pathlib import Path
from datetime import datetime, timezone
from contextlib import redirect_stdout

import numpy as np
import torch


# ============================================================
# 2. PROJECT AND REPOSITORY VERIFICATION
# ============================================================

PROJECT_DIR = Path("/content/Research/projects/01-mini-bart")

assert PROJECT_DIR.exists(), (
    "Mini-BART project directory not found."
)

os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

EXPECTED_COMMIT = (
    "de6924c42989a8ba9094b287d14e89feb2ff7674"
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_DIR,
    text=True,
).strip()

assert actual_commit == EXPECTED_COMMIT, (
    f"Incorrect Git commit.\n"
    f"Expected: {EXPECTED_COMMIT}\n"
    f"Actual: {actual_commit}"
)

assert torch.cuda.is_available(), (
    "CUDA GPU is unavailable. Stopping E2."
)

device = torch.device("cuda")

print("Git commit:", actual_commit)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


# ============================================================
# 3. FINAL EXPERIMENT CONFIGURATION
# ============================================================

NUM_EPOCHS = 2

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

TEMPERATURE = 2.0
LAMBDA_KD = 0.5

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

CHECKPOINT_EVERY_STEPS = 500

SEED = 42

EXPECTED_TRAIN_EXAMPLES = 287113
EXPECTED_VAL_EXAMPLES = 13368

UPDATES_PER_EPOCH = (
    EXPECTED_TRAIN_EXAMPLES
    + GRADIENT_ACCUMULATION_STEPS - 1
) // GRADIENT_ACCUMULATION_STEPS

PLANNED_TOTAL_UPDATES = (
    UPDATES_PER_EPOCH * NUM_EPOCHS
)

assert UPDATES_PER_EPOCH == 35890
assert PLANNED_TOTAL_UPDATES == 71780


# ============================================================
# 4. VERIFY ACTUAL OPTIMIZER AND KD CONSTANTS
# ============================================================

import src.train as train_module
import src.train_kd as train_kd_module

from src.train_kd import get_kd_config

assert train_module.LEARNING_RATE == LEARNING_RATE
assert train_module.WEIGHT_DECAY == WEIGHT_DECAY
assert train_module.WARMUP_RATIO == WARMUP_RATIO

assert train_kd_module.KD_TEMPERATURE == TEMPERATURE
assert train_kd_module.KD_LAMBDA == LAMBDA_KD
assert train_kd_module.KD_CHECKPOINT_EVERY_STEPS == (
    CHECKPOINT_EVERY_STEPS
)
assert train_kd_module.KD_GRADIENT_ACCUMULATION_STEPS == (
    GRADIENT_ACCUMULATION_STEPS
)

EXPECTED_CONFIG = get_kd_config(
    num_epochs=NUM_EPOCHS,
    temperature=TEMPERATURE,
    lambda_kd=LAMBDA_KD,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
)

assert EXPECTED_CONFIG["learning_rate"] == LEARNING_RATE
assert EXPECTED_CONFIG["weight_decay"] == WEIGHT_DECAY
assert EXPECTED_CONFIG["warmup_ratio"] == WARMUP_RATIO

assert EXPECTED_CONFIG["max_input_length"] == MAX_INPUT_LENGTH
assert EXPECTED_CONFIG["max_target_length"] == MAX_TARGET_LENGTH
assert EXPECTED_CONFIG["seed"] == SEED

print("\nOptimizer and KD configuration verified.")
print("Learning rate:", train_module.LEARNING_RATE)
print("Weight decay:", train_module.WEIGHT_DECAY)
print("Warmup ratio:", train_module.WARMUP_RATIO)
print("Maximum gradient norm:", train_module.MAX_GRAD_NORM)


# ============================================================
# 5. VERIFY DATASET
# ============================================================

assert "tokenized_dataset" in globals(), (
    "Full tokenized dataset is not loaded."
)

assert "tokenizer" in globals(), (
    "BART tokenizer is not loaded."
)

assert len(tokenized_dataset["train"]) == EXPECTED_TRAIN_EXAMPLES
assert len(tokenized_dataset["validation"]) == EXPECTED_VAL_EXAMPLES
assert len(tokenized_dataset["test"]) == 0

print("\nDataset verification passed.")
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))
print("Test examples:", len(tokenized_dataset["test"]))


# ============================================================
# 6. GOOGLE DRIVE OUTPUT DIRECTORY
# ============================================================

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/MiniBART_Research"
)

assert DRIVE_PROJECT_DIR.exists(), (
    "Google Drive is not mounted or project folder is missing."
)

FINAL_DIR = DRIVE_PROJECT_DIR / "E2" / "Final"

RUN_DIR = FINAL_DIR / "FullRun_BS1_GA8_Epochs2"

LATEST_CHECKPOINT = (
    RUN_DIR / "latest_kd_checkpoint.pt"
)

BEST_CHECKPOINT = (
    RUN_DIR / "best_kd_checkpoint.pt"
)

RUN_LOG = RUN_DIR / "training_log.txt"

RESULTS_FILE = RUN_DIR / "run_results.json"

print("\nFull-run output directory:")
print(RUN_DIR)


# ============================================================
# 7. CHECKPOINT VALIDATION AND RUN-STATE DETECTION
# ============================================================

RUN_DIR_EXISTS = RUN_DIR.exists()

if LATEST_CHECKPOINT.is_file():

    assert LATEST_CHECKPOINT.stat().st_size > 0, (
        "Latest checkpoint is empty. Stopping safely."
    )

    print("\nInspecting existing recovery checkpoint...")

    saved_checkpoint = torch.load(
        LATEST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = [
        "student_state_dict",
        "epoch",
        "global_step",
        "best_val_loss",
        "config",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "sampler_state_dict",
        "rng_state",
    ]

    for key in required_keys:
        assert key in saved_checkpoint, (
            f"Recovery checkpoint is incomplete: missing {key}"
        )

    assert saved_checkpoint["config"] == EXPECTED_CONFIG, (
        "Existing checkpoint configuration does not match "
        "the current E2 experiment. Stopping without modifying it."
    )

    saved_epoch = saved_checkpoint["epoch"]
    saved_global_step = saved_checkpoint["global_step"]

    assert isinstance(saved_epoch, int)
    assert isinstance(saved_global_step, int)

    assert 0 <= saved_epoch <= NUM_EPOCHS

    assert 0 <= saved_global_step <= PLANNED_TOTAL_UPDATES

    sampler_state = saved_checkpoint["sampler_state_dict"]

    for key in [
        "generator_state",
        "permutation",
        "position",
        "epoch",
    ]:
        assert key in sampler_state, (
            f"Sampler recovery state is incomplete: missing {key}"
        )

    sampler_permutation = sampler_state["permutation"]
    sampler_position = int(sampler_state["position"])
    sampler_epoch = int(sampler_state["epoch"])

    assert sampler_epoch == saved_epoch, (
        "Checkpoint epoch and sampler epoch disagree. "
        "Stopping to protect recovery integrity."
    )

    if sampler_permutation is None:

        assert sampler_position == 0, (
            "Invalid sampler state: position must be zero "
            "when no permutation exists."
        )

    else:

        assert len(sampler_permutation) == EXPECTED_TRAIN_EXAMPLES

        assert 0 <= sampler_position < EXPECTED_TRAIN_EXAMPLES, (
            "Invalid mid-epoch sampler position."
        )

    if saved_epoch == NUM_EPOCHS:

        assert sampler_permutation is None
        assert sampler_position == 0

        assert BEST_CHECKPOINT.is_file(), (
            "Training checkpoint indicates completion, "
            "but the best-model checkpoint is missing. "
            "Stopping for artifact inspection."
        )

        assert BEST_CHECKPOINT.stat().st_size > 0

        RUN_MODE = "ALREADY_COMPLETED"
        RESUME_FROM = None

    else:

        RUN_MODE = "RESUME"
        RESUME_FROM = str(LATEST_CHECKPOINT)

        print("Valid recovery checkpoint detected.")
        print("Saved next epoch:", saved_epoch)
        print("Saved optimizer updates:", saved_global_step)

elif RUN_DIR_EXISTS:

    existing_files = list(RUN_DIR.iterdir())

    if existing_files:

        raise RuntimeError(
            "The full-run directory contains existing artifacts, "
            "but latest_kd_checkpoint.pt is missing.\n"
            "No files have been modified.\n"
            "Inspect this directory before proceeding:\n"
            f"{RUN_DIR}"
        )

    RUN_MODE = "FRESH"
    RESUME_FROM = None

else:

    RUN_MODE = "FRESH"
    RESUME_FROM = None


# ============================================================
# 8. HANDLE ALREADY-COMPLETED RUN
# ============================================================

if RUN_MODE == "ALREADY_COMPLETED":

    print("\n==========================================")
    print("E2 FULL TRAINING ALREADY COMPLETED")
    print("==========================================")

    print("Completed epochs:", saved_epoch)
    print("Successful optimizer updates:", saved_global_step)
    print("Latest checkpoint:", LATEST_CHECKPOINT)
    print("Best checkpoint:", BEST_CHECKPOINT)

    if RESULTS_FILE.is_file():
        print("Run results:", RESULTS_FILE)
    else:
        print(
            "Warning: run_results.json is missing. "
            "Training itself is complete; inspect the log "
            "and checkpoints before reconstructing metadata."
        )

    print("\nNo training was launched.")
    print("No checkpoint or result file was overwritten.")

else:

    # ========================================================
    # 9. SET SEEDS BEFORE DATA LOADER AND MODEL CREATION
    # ========================================================

    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    print("\nRandom seeds initialized:", SEED)


    # ========================================================
    # 10. CREATE FULL-RUN DATALOADERS
    # ========================================================

    from src.data import create_dataloaders

    train_loader, val_loader, test_loader = create_dataloaders(
        tokenized_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        batch_size=BATCH_SIZE,
        num_workers=0,
    )

    assert len(train_loader) == EXPECTED_TRAIN_EXAMPLES
    assert len(val_loader) == EXPECTED_VAL_EXAMPLES
    assert len(test_loader) == 0

    assert train_loader.batch_size == BATCH_SIZE
    assert train_loader.num_workers == 0

    assert hasattr(train_loader.sampler, "state_dict")
    assert hasattr(train_loader.sampler, "load_state_dict")

    print("\nDataLoader verification passed.")
    print("Training microbatches:", len(train_loader))
    print("Validation batches:", len(val_loader))
    print("Test batches:", len(test_loader))

    print(
        "Effective batch size:",
        BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    )


    # ========================================================
    # 11. CLEAN UP OLD CELL 1 / PILOT MODEL REFERENCES
    # ========================================================

    # Do not reuse a potentially modified pilot or preflight student.
    # A fresh teacher-derived student is constructed below.

    for variable_name in [
        "student_model",
        "recovery_student",
        "resumed_student",
        "teacher_model",
        "teacher",
        "base_teacher",
    ]:
        globals().pop(variable_name, None)

    gc.collect()
    torch.cuda.empty_cache()


    # ========================================================
    # 12. LOAD FROZEN TEACHER AND CREATE FRESH STUDENT
    # ========================================================

    from src.model import (
        load_teacher_model,
        prepare_kd_models,
    )

    print("\nLoading pretrained BART teacher...")

    base_teacher = load_teacher_model()

    base_teacher = base_teacher.to(
        device=device,
        dtype=torch.float16,
    )

    base_teacher.eval()

    for parameter in base_teacher.parameters():
        parameter.requires_grad = False

    teacher_model, student_model = prepare_kd_models(
        base_teacher
    )

    teacher_model = teacher_model.to(device)

    student_model = student_model.to(device)

    assert teacher_model.config.encoder_layers == 12
    assert teacher_model.config.decoder_layers == 12

    assert student_model.config.encoder_layers == 6
    assert student_model.config.decoder_layers == 6

    assert all(
        not parameter.requires_grad
        for parameter in teacher_model.parameters()
    )

    assert all(
        parameter.requires_grad
        for parameter in student_model.parameters()
    )

    assert teacher_model.training is False
    assert student_model.training is True

    assert (
        student_model.model.shared.weight.data_ptr()
        == student_model.lm_head.weight.data_ptr()
    )

    print("\nTeacher and student verification passed.")
    print("Teacher: frozen, FP16, 12E + 12D")
    print("Student: trainable, 6E + 6D")
    print("Fresh teacher-derived initialization: YES")


    # ========================================================
    # 13. TRAINING LOG SUPPORT
    # ========================================================

    class StdoutTee:

        def __init__(self, *streams):
            self.streams = streams

        def write(self, message):
            for stream in self.streams:
                stream.write(message)
                stream.flush()
            return len(message)

        def flush(self):
            for stream in self.streams:
                stream.flush()


    # ========================================================
    # 14. FINAL CONFIGURATION DISPLAY
    # ========================================================

    RUN_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("\n========== FINAL E2 CONFIGURATION ==========")

    print("Run mode:", RUN_MODE)

    print("Epochs:", NUM_EPOCHS)

    print("Training examples:", EXPECTED_TRAIN_EXAMPLES)
    print("Validation examples:", EXPECTED_VAL_EXAMPLES)

    print("Per-device batch size:", BATCH_SIZE)
    print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

    print(
        "Effective batch size:",
        BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    )

    print("Updates per epoch:", UPDATES_PER_EPOCH)
    print("Planned total updates:", PLANNED_TOTAL_UPDATES)

    print("Learning rate:", LEARNING_RATE)
    print("Weight decay:", WEIGHT_DECAY)
    print("Warmup ratio:", WARMUP_RATIO)

    print("KD temperature:", TEMPERATURE)
    print("KD lambda:", LAMBDA_KD)

    print("Checkpoint interval:", CHECKPOINT_EVERY_STEPS)
    print("Maximum input length:", MAX_INPUT_LENGTH)
    print("Maximum target length:", MAX_TARGET_LENGTH)

    print("Seed:", SEED)

    print("Resume checkpoint:", RESUME_FROM)

    print("============================================")


    # ========================================================
    # 15. LAUNCH OR RESUME TRAINING
    # ========================================================

    from src.train_kd import train_kd_model

    torch.cuda.reset_peak_memory_stats()

    session_started = datetime.now(timezone.utc).isoformat()

    with open(
        RUN_LOG,
        "a",
        encoding="utf-8",
        buffering=1,
    ) as log_file:

        with redirect_stdout(
            StdoutTee(sys.stdout, log_file)
        ):

            print("\n\n========================================")
            print("E2 FULL TRAINING SESSION")
            print("Started:", session_started)
            print("Run mode:", RUN_MODE)
            print("========================================")

            e2_results = train_kd_model(
                teacher=teacher_model,
                student=student_model,

                train_dataloader=train_loader,
                val_dataloader=val_loader,

                device=device,

                output_dir=str(RUN_DIR),

                num_epochs=NUM_EPOCHS,

                temperature=TEMPERATURE,
                lambda_kd=LAMBDA_KD,

                gradient_accumulation_steps=(
                    GRADIENT_ACCUMULATION_STEPS
                ),

                checkpoint_every_steps=(
                    CHECKPOINT_EVERY_STEPS
                ),

                resume_from=RESUME_FROM,
            )

            print("\nE2 training function returned.")


    # ========================================================
    # 16. VERIFY FINAL CHECKPOINT BEFORE RECORDING COMPLETION
    # ========================================================

    assert LATEST_CHECKPOINT.is_file()
    assert BEST_CHECKPOINT.is_file()

    assert LATEST_CHECKPOINT.stat().st_size > 0
    assert BEST_CHECKPOINT.stat().st_size > 0

    final_checkpoint = torch.load(
        LATEST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    assert final_checkpoint["config"] == EXPECTED_CONFIG

    assert final_checkpoint["epoch"] == NUM_EPOCHS, (
        "Training function returned, but the latest checkpoint "
        "does not confirm completion of both epochs."
    )

    assert (
        final_checkpoint["global_step"]
        == e2_results["global_step"]
    )

    assert final_checkpoint["best_val_loss"] == (
        e2_results["best_val_loss"]
    )

    assert e2_results["global_step"] <= PLANNED_TOTAL_UPDATES

    print("\nFinal checkpoint verification passed.")
    print("Completed epochs:", final_checkpoint["epoch"])
    print("Successful optimizer updates:", e2_results["global_step"])


    # ========================================================
    # 17. SAVE FINAL RUN RECORD ATOMICALLY
    # ========================================================

    peak_gpu_memory_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    run_record = {

        "experiment": "E2 Mini-BART Knowledge Distillation",

        "run_type": "Full",

        "status": "completed",

        "github_repository": "HeavenlyDemon27/Research",
        "github_commit": actual_commit,

        "teacher": "facebook/bart-large-cnn",

        "student_architecture": "6 encoder + 6 decoder layers",

        "teacher_precision": "float16_autocast",
        "student_training_precision": "float16_autocast_with_fp32_parameters",

        "dataset": "abisee/cnn_dailymail",
        "dataset_version": "3.0.0",

        "training_examples": EXPECTED_TRAIN_EXAMPLES,
        "validation_examples": EXPECTED_VAL_EXAMPLES,
        "test_examples_used_for_training": 0,

        "epochs": NUM_EPOCHS,

        "per_device_train_batch_size": BATCH_SIZE,

        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),

        "effective_batch_size": (
            BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
        ),

        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,

        "temperature": TEMPERATURE,
        "lambda_kd": LAMBDA_KD,

        "max_input_length": MAX_INPUT_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,

        "planned_optimizer_updates": PLANNED_TOTAL_UPDATES,

        "successful_optimizer_updates": (
            e2_results["global_step"]
        ),

        "best_validation_loss": (
            e2_results["best_val_loss"]
        ),

        "checkpoint_every_optimizer_steps": (
            CHECKPOINT_EVERY_STEPS
        ),

        "seed": SEED,

        "run_mode_this_session": RUN_MODE,

        "session_started_at_utc": session_started,

        "completed_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "history_this_session": e2_results["history"],

        "latest_checkpoint": str(LATEST_CHECKPOINT),

        "best_checkpoint": str(BEST_CHECKPOINT),

        "training_log": str(RUN_LOG),

        "peak_gpu_memory_gb": peak_gpu_memory_gb,
    }

    temporary_results = Path(
        str(RESULTS_FILE) + ".tmp"
    )

    with open(
        temporary_results,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            run_record,
            f,
            indent=4,
        )

    os.replace(
        temporary_results,
        RESULTS_FILE,
    )


    # ========================================================
    # 18. FINAL SUMMARY
    # ========================================================

    print("\n========== E2 TRAINING COMPLETED ==========")

    print("Run mode:", RUN_MODE)

    print("Completed epochs:", NUM_EPOCHS)

    print(
        "Successful optimizer updates:",
        e2_results["global_step"],
    )

    print(
        "Planned optimizer updates:",
        PLANNED_TOTAL_UPDATES,
    )

    print(
        "Best validation loss:",
        e2_results["best_val_loss"],
    )

    print(
        "Peak GPU memory:",
        round(peak_gpu_memory_gb, 2),
        "GB",
    )

    print("\nLatest checkpoint:", LATEST_CHECKPOINT)
    print("Best checkpoint:", BEST_CHECKPOINT)
    print("Training log:", RUN_LOG)
    print("Results record:", RESULTS_FILE)

    print("\n===========================================")
